### STAGE 0

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!find "/content/" -name "*.zip" -exec unzip -o {} -d "/content/output/" \;


Archive:  /content/smiling_left_3quarter.zip
   creating: /content/output/smiling_left_3quarter/
  inflating: /content/output/smiling_left_3quarter/.DS_Store  
   creating: /content/output/__MACOSX/
   creating: /content/output/__MACOSX/smiling_left_3quarter/
  inflating: /content/output/__MACOSX/smiling_left_3quarter/._.DS_Store  
  inflating: /content/output/smiling_left_3quarter/001_07.jpg  
  inflating: /content/output/smiling_left_3quarter/002_07.jpg  
  inflating: /content/output/smiling_left_3quarter/003_07.jpg  
  inflating: /content/output/smiling_left_3quarter/004_07.jpg  
  inflating: /content/output/smiling_left_3quarter/005_07.jpg  
  inflating: /content/output/smiling_left_3quarter/006_07.jpg  
  inflating: /content/output/smiling_left_3quarter/007_07.jpg  
  inflating: /content/output/smiling_left_3quarter/008_07.jpg  
  inflating: /content/output/smiling_left_3quarter/009_07.jpg  
  inflating: /content/output/smiling_left_3quarter/010_07.jpg  
  inflating: /content/outp

In [13]:
%%writefile prepare.py
#!/usr/bin/env python3
"""
prepare.py - GALNP Stage 0: face preprocessing (MediaPipe + OpenCV + NumPy only).

For every frontal image (neutral_front/, smiling_front/) this script:
  1. runs the MediaPipe Tasks Vision FaceLandmarker (478 landmarks, 52 blendshapes,
     4x4 facial transformation matrix),
  2. aligns the face to a canonical 512x512 template with a 5-point similarity transform,
  3. saves the aligned RGB crop (JPEG, quality 95) and a metadata .npz file.

This version adds an automatic per-image "zoom out" step so the face (forehead -> chin)
occupies a configurable fraction of the vertical crop (recommended 0.70-0.80).
If the face is too large, the canonical template is scaled inward about the crop
center to produce more forehead, chin/neck and left/right margins. Scaling is clamped
to avoid extreme zooming out. Only zoom-out is performed (no artificial zoom-in).
"""

from __future__ import annotations

import argparse
import csv
import logging
import urllib.request
from collections import Counter
from pathlib import Path
from typing import Dict, List, NamedTuple, Optional, Tuple

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm import tqdm

LOGGER = logging.getLogger("prepare")

# --------------------------------------------------------------------------- #
# Constants
# --------------------------------------------------------------------------- #
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
    "face_landmarker_v2_with_blendshapes/float16/1/face_landmarker_v2_with_blendshapes.task"
)
DEFAULT_MODEL_PATH = Path("data/models/face_landmarker_v2_with_blendshapes.task")

NUM_LANDMARKS = 478
NUM_BLENDSHAPES = 52

# Folder name -> expression label
EXPRESSION_DIRS: Dict[str, str] = {
    "neutral_front": "neutral",
    "smiling_front": "smiling",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# 5 MediaPipe landmark indices, ordered left-to-right *as seen in the image*:
#   33  = outer corner of the eye on the image-left  (subject's right eye)
#   263 = outer corner of the eye on the image-right (subject's left eye)
#   1   = nose tip
#   61  = mouth corner on the image-left
#   291 = mouth corner on the image-right
FIVE_POINT_IDX: Tuple[int, ...] = (33, 263, 1, 61, 291)

# Landmarks used only for framing diagnostics (not for alignment):
FOREHEAD_TOP_IDX = 10   # top of the face oval (about the hairline)
CHIN_IDX = 152          # bottom of the chin

# Canonical template in normalized [0, 1] crop coordinates (same order as above).
CANONICAL_TEMPLATE_NORM = np.array(
    [
        [0.305, 0.39],  # eye outer corner (image-left)
        [0.695, 0.39],  # eye outer corner (image-right)
        [0.500, 0.55],  # nose tip
        [0.381, 0.68],  # mouth corner (image-left)
        [0.619, 0.68],  # mouth corner (image-right)
    ],
    dtype=np.float64,
)


class CsvMeta(NamedTuple):
    age: float
    gender: str
    ethnicity: str


EMPTY_META = CsvMeta(age=float("nan"), gender="", ethnicity="")

class Outcome(NamedTuple):
    success: bool
    reason: str
    extent: Optional[Tuple[float, float]] = None  # (forehead_top_y, chin_y) as crop fractions

class FaceResult(NamedTuple):
    landmarks: np.ndarray     # (478, 3) normalized image coords (x, y in [0,1], z ~ x scale)
    blendshapes: np.ndarray   # (52,)
    pose: np.ndarray          # (4, 4)


# --------------------------------------------------------------------------- #
# Model loading
# --------------------------------------------------------------------------- #
def ensure_model(model_path: Path) -> Path:
    """Download the FaceLandmarker model if it is not already on disk."""
    if model_path.exists():
        return model_path
    model_path.parent.mkdir(parents=True, exist_ok=True)
    LOGGER.info("Downloading FaceLandmarker model to %s ...", model_path)
    try:
        urllib.request.urlretrieve(MODEL_URL, model_path)
    except Exception as exc:  # noqa: BLE001
        if model_path.exists():
            model_path.unlink()
        raise RuntimeError(
            f"Could not download the model from {MODEL_URL}.\n"
            f"Download it manually and pass --model_path. Error: {exc}"
        ) from exc
    return model_path


def load_landmarker(model_path: Path, min_confidence: float) -> vision.FaceLandmarker:
    """Create a FaceLandmarker (IMAGE mode) with blendshapes + transformation matrices."""
    options = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(model_path)),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=min_confidence,
        min_face_presence_confidence=min_confidence,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return vision.FaceLandmarker.create_from_options(options)


# --------------------------------------------------------------------------- #
# CSV metadata
# --------------------------------------------------------------------------- #
def _normalize_id(raw: str) -> str:
    """'001', '1', ' 001.jpg ' -> '1' (numeric IDs are compared as integers)."""
    s = Path(str(raw).strip()).stem if str(raw).strip().lower().endswith(tuple(IMAGE_EXTENSIONS)) else str(raw).strip()
    return str(int(s)) if s.isdigit() else s


def find_csv(input_dir: Path, csv_path: Optional[Path]) -> Optional[Path]:
    """Use the given CSV, else look for a ratings CSV inside the input directory."""
    if csv_path is not None:
        return csv_path if csv_path.exists() else None
    preferred = input_dir / "london_faces_ratings.csv"
    if preferred.exists():
        return preferred
    candidates = sorted(input_dir.glob("*.csv"))
    return candidates[0] if candidates else None


def load_metadata(csv_path: Optional[Path]) -> Dict[str, CsvMeta]:
    """Load face_id -> (age, gender, ethnicity). Returns {} if the CSV is unavailable."""
    if csv_path is None:
        LOGGER.warning("No CSV found; age/gender/ethnicity will be left empty.")
        return {}

    table: Dict[str, CsvMeta] = {}
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None or "face_id" not in reader.fieldnames:
            LOGGER.warning("CSV %s has no 'face_id' column; metadata ignored.", csv_path)
            return {}
        for row in reader:
            face_id = (row.get("face_id") or "").strip()
            if not face_id:
                continue
            try:
                age = float((row.get("face_age") or "").strip())
            except ValueError:
                age = float("nan")
            table[_normalize_id(face_id)] = CsvMeta(
                age=age,
                gender=(row.get("face_gender") or "").strip(),
                ethnicity=(row.get("face_eth") or "").strip(),
            )
    LOGGER.info("Loaded metadata for %d faces from %s", len(table), csv_path)
    return table


# --------------------------------------------------------------------------- #
# Landmark detection
# --------------------------------------------------------------------------- #
def detect_face(landmarker: vision.FaceLandmarker, rgb: np.ndarray) -> Optional[FaceResult]:
    """Run FaceLandmarker on an RGB uint8 image. Returns None if no face is found."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        return None
    if not result.face_blendshapes or not result.facial_transformation_matrixes:
        return None

    lm = result.face_landmarks[0]
    if len(lm) < NUM_LANDMARKS:
        return None
    landmarks = np.array([[p.x, p.y, p.z] for p in lm[:NUM_LANDMARKS]], dtype=np.float32)

    cats = sorted(result.face_blendshapes[0], key=lambda c: c.index)
    blendshapes = np.array([c.score for c in cats], dtype=np.float32)
    if blendshapes.shape[0] != NUM_BLENDSHAPES:
        return None

    pose = np.asarray(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(4, 4)
    return FaceResult(landmarks, blendshapes, pose)


def compute_confidence(landmarks_norm: np.ndarray) -> float:
    """Fraction of landmarks inside the image frame (proxy confidence, see module docs)."""
    xy = landmarks_norm[:, :2]
    inside = (xy[:, 0] >= 0.0) & (xy[:, 0] <= 1.0) & (xy[:, 1] >= 0.0) & (xy[:, 1] <= 1.0)
    return float(inside.mean())


def get_5point_landmarks(landmarks_norm: np.ndarray, width: int, height: int) -> np.ndarray:
    """Return the 5 alignment points in original-image pixel coordinates, shape (5, 2)."""
    pts = landmarks_norm[list(FIVE_POINT_IDX), :2].astype(np.float64)
    pts[:, 0] *= width
    pts[:, 1] *= height
    return pts


# --------------------------------------------------------------------------- #
# Alignment and auto-zoom helpers
# --------------------------------------------------------------------------- #
def estimate_similarity_transform(src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    """
    Least-squares 2D similarity transform (rotation + uniform scale + translation)
    mapping `src` -> `dst` (Umeyama, 1991). Both are (N, 2). Returns a (2, 3) matrix.
    """
    n = src.shape[0]
    src_mean, dst_mean = src.mean(axis=0), dst.mean(axis=0)
    src_c, dst_c = src - src_mean, dst - dst_mean

    cov = dst_c.T @ src_c / n
    u, s, vt = np.linalg.svd(cov)
    d = np.ones(2)
    if np.linalg.det(u) * np.linalg.det(vt) < 0:
        d[-1] = -1.0
    rot = u @ np.diag(d) @ vt

    var_src = (src_c ** 2).sum() / n
    scale = float((s * d).sum() / var_src)
    trans = dst_mean - scale * rot @ src_mean
    return np.hstack([scale * rot, trans[:, None]])


def align_and_crop(
    image_rgb: np.ndarray,
    landmarks_norm: np.ndarray,
    target_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Align the face to the canonical template.

    Returns:
        crop:    (target_size, target_size, 3) uint8 RGB
        matrix:  (2, 3) float64 affine matrix mapping original pixels -> crop pixels
    """
    h, w = image_rgb.shape[:2]
    src = get_5point_landmarks(landmarks_norm, w, h)
    dst = CANONICAL_TEMPLATE_NORM * target_size
    matrix = estimate_similarity_transform(src, dst)
    crop = cv2.warpAffine(
        image_rgb,
        matrix,
        (target_size, target_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return crop, matrix


def transform_landmarks(landmarks_norm: np.ndarray, matrix: np.ndarray, width: int, height: int) -> np.ndarray:
    """Map normalized landmarks into aligned-crop pixel coordinates, (478, 3) float32."""
    xy = landmarks_norm[:, :2].astype(np.float64) * np.array([width, height])
    xy_crop = xy @ matrix[:, :2].T + matrix[:, 2]
    scale = float(np.sqrt(abs(np.linalg.det(matrix[:, :2]))))
    z_crop = landmarks_norm[:, 2].astype(np.float64) * width * scale  # MediaPipe z ~ x scale
    return np.concatenate([xy_crop, z_crop[:, None]], axis=1).astype(np.float32)


def face_vertical_extent(landmarks_crop: np.ndarray, target_size: int) -> Tuple[float, float]:
    """(forehead_top_y, chin_y) of the face oval as fractions of the crop height."""
    return (
        float(landmarks_crop[FOREHEAD_TOP_IDX, 1]) / target_size,
        float(landmarks_crop[CHIN_IDX, 1]) / target_size,
    )


def compute_zoom_scale_for_face_from_crop(
    landmarks_crop: np.ndarray,
    target_size: int,
    desired_frac: float = 0.75,
    min_scale: float = 1.0,
    max_scale: float = 2.0,
) -> float:
    """
    landmarks_crop: (N,3) landmarks already transformed into crop pixel coords.
    Returns scale >= 1.0 to apply to the canonical template (scale > 1 => zoom out).
    """
    forehead_y = float(landmarks_crop[FOREHEAD_TOP_IDX, 1])
    chin_y = float(landmarks_crop[CHIN_IDX, 1])
    current_frac = (chin_y - forehead_y) / float(target_size)
    if current_frac <= 0 or not np.isfinite(current_frac):
        return 1.0
    # Desired scale to make current_frac * (1/scale) == desired_frac  => scale = current_frac / desired_frac
    scale = current_frac / desired_frac
    # We only allow zoom-out (scale >= 1.0). If current_frac < desired_frac, scale < 1 => keep 1.0
    if scale < min_scale:
        scale = min_scale
    if scale > max_scale:
        scale = max_scale
    return float(scale)


def align_and_crop_autozoom(
    image_rgb: np.ndarray,
    landmarks_norm: np.ndarray,
    target_size: int,
    desired_frac: float = 0.75,
    max_zoom_out: float = 1.6,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Align the face to the canonical template, but automatically zoom out if the face
    occupies more than desired_frac of the crop height.

    Returns:
        crop: (target_size, target_size, 3) uint8 RGB
        matrix: (2, 3) float64 affine matrix mapping original pixels -> crop pixels
        applied_scale: float (>=1.0) applied to the canonical template (1.0 = no zoom)
    """
    h, w = image_rgb.shape[:2]
    src = get_5point_landmarks(landmarks_norm, w, h)
    dst = CANONICAL_TEMPLATE_NORM * target_size
    matrix = estimate_similarity_transform(src, dst)

    # Transform landmarks into crop coords to measure current face extent
    landmarks_crop = transform_landmarks(landmarks_norm, matrix, w, h)

    # Compute scale factor to make face height ~ desired_frac of crop
    scale = compute_zoom_scale_for_face_from_crop(landmarks_crop, target_size, desired_frac, min_scale=1.0, max_scale=max_zoom_out)

    if scale == 1.0:
        crop = cv2.warpAffine(
            image_rgb,
            matrix,
            (target_size, target_size),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(0, 0, 0),
        )
        return crop, matrix, scale

    # Scale the canonical template about the crop center so that destination points move inward
    center = np.array([target_size / 2.0, target_size / 2.0], dtype=np.float64)
    dst_scaled = center + (dst - center) / scale

    # Recompute transform mapping src -> dst_scaled
    matrix_scaled = estimate_similarity_transform(src, dst_scaled)
    crop_scaled = cv2.warpAffine(
        image_rgb,
        matrix_scaled,
        (target_size, target_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return crop_scaled, matrix_scaled, scale


# --------------------------------------------------------------------------- #
# Per-image processing
# --------------------------------------------------------------------------- #
def process_image(
    image_path: Path,
    expression: str,
    face_id: str,
    meta: CsvMeta,
    landmarker: vision.FaceLandmarker,
    crops_dir: Path,
    meta_dir: Path,
    target_size: int,
    min_confidence: float,
    desired_frac: float,
    max_zoom_out: float,
) -> Outcome:
    """Process a single image. Returns an Outcome(success, reason, face extent)."""
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return Outcome(False, "unreadable_image")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]

    face = detect_face(landmarker, rgb)
    if face is None:
        return Outcome(False, "no_face_detected")

    confidence = compute_confidence(face.landmarks)
    if confidence < min_confidence:
        return Outcome(False, "low_confidence")

    five = get_5point_landmarks(face.landmarks, w, h)
    if not np.isfinite(five).all():
        return Outcome(False, "invalid_landmarks")

    # Auto-zoom alignment: try to make face occupy desired_frac of the crop height
    crop_rgb, matrix, applied_scale = align_and_crop_autozoom(rgb, face.landmarks, target_size, desired_frac, max_zoom_out)
    landmarks_crop = transform_landmarks(face.landmarks, matrix, w, h)

    stem = f"{face_id}_{expression}"
    ok = cv2.imwrite(
        str(crops_dir / f"{stem}.jpg"),
        cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR),
        [cv2.IMWRITE_JPEG_QUALITY, 95],
    )
    if not ok:
        return Outcome(False, "write_failed")

    np.savez_compressed(
        meta_dir / f"{stem}.npz",
        landmarks=landmarks_crop,
        blendshapes=face.blendshapes.astype(np.float32),
        pose=face.pose.astype(np.float32),
        confidence=float(confidence),
        expression=expression,
        face_id=face_id,
        age=float(meta.age),
        gender=meta.gender,
        ethnicity=meta.ethnicity,
        align_matrix=matrix.astype(np.float32),
        target_size=int(target_size),
        applied_scale=float(applied_scale),
    )
    return Outcome(True, "ok", face_vertical_extent(landmarks_crop, target_size))


# --------------------------------------------------------------------------- #
# Dataset traversal
# --------------------------------------------------------------------------- #
def collect_images(input_dir: Path) -> List[Tuple[Path, str, str]]:
    """Return a list of (image_path, expression, face_id) for all frontal images."""
    items: List[Tuple[Path, str, str]] = []
    for folder, expression in EXPRESSION_DIRS.items():
        folder_path = input_dir / folder
        if not folder_path.is_dir():
            LOGGER.warning("Folder not found, skipping: %s", folder_path)
            continue
        files = sorted(p for p in folder_path.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
        LOGGER.info("Found %d images in %s", len(files), folder_path)
        items.extend((p, expression, p.stem) for p in files)
    return items


def run(
    input_dir: Path,
    output_dir: Path,
    csv_path: Optional[Path],
    target_size: int,
    min_confidence: float,
    model_path: Path = DEFAULT_MODEL_PATH,
    overwrite: bool = False,
    desired_frac: float = 0.75,
    max_zoom_out: float = 1.6,
) -> Counter:
    """Run Stage 0 preprocessing over the whole dataset. Returns a Counter of outcomes."""
    crops_dir = output_dir / "crops"
    meta_dir = output_dir / "meta"
    crops_dir.mkdir(parents=True, exist_ok=True)
    meta_dir.mkdir(parents=True, exist_ok=True)

    metadata = load_metadata(find_csv(input_dir, csv_path))
    items = collect_images(input_dir)
    if not items:
        raise FileNotFoundError(
            f"No images found. Expected {list(EXPRESSION_DIRS)} inside {input_dir}"
        )

    landmarker = load_landmarker(ensure_model(model_path), min_confidence)
    stats: Counter = Counter()
    missing_csv = 0
    extents: List[Tuple[float, float]] = []
    scales: List[float] = []

    try:
        for image_path, expression, face_id in tqdm(items, desc="Preprocessing", unit="img"):
            stem = f"{face_id}_{expression}"
            if not overwrite and (crops_dir / f"{stem}.jpg").exists() and (meta_dir / f"{stem}.npz").exists():
                stats["skipped_existing"] += 1
                continue

            meta = metadata.get(_normalize_id(face_id))
            if meta is None:
                meta = EMPTY_META
                missing_csv += 1

            try:
                outcome = process_image(
                    image_path, expression, face_id, meta, landmarker,
                    crops_dir, meta_dir, target_size, min_confidence,
                    desired_frac, max_zoom_out,
                )
            except Exception as exc:  # noqa: BLE001 - never let one bad image kill the run
                LOGGER.debug("Error on %s: %s", image_path, exc)
                outcome = Outcome(False, "exception")

            success, reason = outcome.success, outcome.reason
            if success:
                stats["success"] += 1
                if outcome.extent is not None:
                    extents.append(outcome.extent)
                    if outcome.extent[0] < 0.0 or outcome.extent[1] > 1.0:
                        stats["warn_face_clipped"] += 1
                # Read applied_scale from the saved npz to collect stats (optional)
                try:
                    npz = np.load(meta_dir / f"{stem}.npz")
                    applied_scale = float(npz.get("applied_scale", 1.0))
                    scales.append(applied_scale)
                except Exception:
                    pass
            else:
                stats["failed"] += 1
                stats[f"failed_{reason}"] += 1
                LOGGER.debug("Skipped %s (%s)", image_path.name, reason)
    finally:
        landmarker.close()

    total = len(items)
    LOGGER.info("=" * 50)
    LOGGER.info("Total images       : %d", total)
    LOGGER.info("Succeeded          : %d", stats["success"])
    LOGGER.info("Failed             : %d", stats["failed"])
    for key in sorted(k for k in stats if k.startswith("failed_")):
        LOGGER.info("  - %-22s: %d", key.replace("failed_", ""), stats[key])
    if extents:
        top, chin = np.mean(extents, axis=0)
        LOGGER.info(
            "Framing (mean)     : forehead-top y=%.2f, chin y=%.2f -> face height %.0f%%, "
            "space below chin %.0f%%",
            top, chin, 100 * (chin - top), 100 * (1 - chin),
        )
        if stats["warn_face_clipped"]:
            LOGGER.warning("%d crops have the forehead top or chin outside the frame",
                           stats["warn_face_clipped"])
    if scales:
        mean_scale = float(np.mean(scales))
        LOGGER.info("Applied scale (mean): %.3f (1.0 = no zoom-out)", mean_scale)
    if stats["skipped_existing"]:
        LOGGER.info("Already processed  : %d (use --overwrite to redo)", stats["skipped_existing"])
    if metadata and missing_csv:
        LOGGER.warning("%d images had no matching CSV row (metadata left empty)", missing_csv)
    LOGGER.info("Output             : %s", output_dir.resolve())
    return stats


# --------------------------------------------------------------------------- #
# CLI
# --------------------------------------------------------------------------- #
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="GALNP Stage 0: MediaPipe FaceLandmarker preprocessing and 5-point alignment.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--input_dir", type=Path, required=True,
                   help="Root folder containing neutral_front/ and smiling_front/")
    p.add_argument("--output_dir", type=Path, default=Path("data/processed"),
                   help="Where crops/ and meta/ are written")
    p.add_argument("--csv_path", type=Path, default=None,
                   help="Ratings CSV with face_id, face_age, face_gender, face_eth "
                        "(auto-detected in input_dir if omitted)")
    p.add_argument("--target_size", type=int, default=512, help="Aligned crop size in pixels")
    p.add_argument("--min_confidence", type=float, default=0.5,
                   help="Minimum detection/presence confidence (and in-frame landmark fraction)")
    p.add_argument("--model_path", type=Path, default=DEFAULT_MODEL_PATH,
                   help="FaceLandmarker .task file (downloaded automatically if missing)")
    p.add_argument("--overwrite", action="store_true", help="Reprocess images that already have outputs")
    p.add_argument("--verbose", action="store_true", help="Log every skipped image")
    p.add_argument("--target_face_frac", type=float, default=0.75,
                   help="Desired fraction of crop height occupied by face (0.7-0.8 recommended)")
    p.add_argument("--max_zoom_out", type=float, default=1.6,
                   help="Maximum allowed zoom-out scale (safety cap)")
    return p.parse_args(argv)


def main(argv: Optional[List[str]] = None) -> None:
    args = parse_args(argv)
    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s | %(levelname)-7s | %(message)s",
        datefmt="%H:%M:%S",
    )
    run(
        input_dir=args.input_dir,
        output_dir=args.output_dir,
        csv_path=args.csv_path,
        target_size=args.target_size,
        min_confidence=args.min_confidence,
        model_path=args.model_path,
        overwrite=args.overwrite,
        desired_frac=args.target_face_frac,
        max_zoom_out=args.max_zoom_out,
    )


if __name__ == "__main__":
    # Usage examples
    # --------------
    # Local:
    #   pip install mediapipe opencv-python numpy tqdm
    #   python prepare.py --input_dir /path/to/london_faces \
    #                     --csv_path /path/to/london_faces/london_faces_ratings.csv \
    #                     --output_dir data/processed \
    #                     --target_face_frac 0.75 --max_zoom_out 1.6
    #
    # Google Colab:
    #   !pip install -q mediapipe opencv-python-headless tqdm
    #   !python prepare.py --input_dir /content/drive/MyDrive/london_faces \
    #                      --output_dir /content/drive/MyDrive/galnp/processed \
    #                      --target_face_frac 0.75
    main()


Overwriting prepare.py


In [4]:
!pip install -q mediapipe opencv-python-headless tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 41.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.7 MB/s eta 0:00:00


In [53]:
!python prepare.py --input_dir /content/output/ \
                          --output_dir /content/drive/MyDrive/galnp/processed --overwrite

2026-10-01 13:35:18.979783: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-01 13:35:19.061804: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
13:35:25 | WARNING | CSV /content/output/london_faces_ratings.csv has no 'face_id' column; metadata ignored.
13:35:25 | INFO    | Found 102 images in /content/output/neutral_front
13:35:25 | INFO    | Found 102 images in /content/output/smiling_front
W0000 00:00:1790861725.378891   27701 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1790861725.398424   27706 inference_feedback_manager.cc:121] Feedback manager requires a mo

In [16]:
%%writefile template.py

"""
geometry/template.py

Utilities for loading and preparing the canonical MediaPipe face template
for GALNP Stage 0.

Features
- Load an OBJ file containing the canonical face (verts, faces, UVs, groups).
- Optional automatic download of a canonical_face_model.obj from a URL.
- One-pass midpoint subdivision (each triangle -> 4 triangles) producing
  ~1.8k verts / ~3.6k faces from a 468-vertex base mesh.
- Preserve and propagate UVs and per-vertex region labels (if present as
  face groups / OBJ 'g' or 'usemtl' entries).
- Build a left-right mirror vertex index map (by mirrored X coordinate).
- Identify mouth cavity faces from a named group or by heuristic (inner lip loop).
- Save / load prepared template to a compact .npz file.
- Small visualization helper to plot the mesh colored by region.

Dependencies: numpy, scipy, trimesh, matplotlib
(Only NumPy / SciPy / trimesh are required by the user; matplotlib is optional
for the visualization helper.)
"""

from __future__ import annotations

import io
import os
import pickle
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import trimesh
from scipy.spatial import cKDTree

# Optional import for visualization
try:
    import matplotlib.pyplot as plt  # type: ignore
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    _HAS_MPL = True
except Exception:
    _HAS_MPL = False


# ---------------------------
# Dataclass for the template
# ---------------------------
@dataclass
class MeshTemplate:
    """
    Container for a prepared canonical face template.

    Attributes
    ----------
    verts : np.ndarray
        (V, 3) float32 vertex positions in canonical coordinates.
    faces : np.ndarray
        (F, 3) int32 triangle indices.
    uvs : np.ndarray
        (V, 2) float32 per-vertex UV coordinates (if available). If not present,
        this will be an array of zeros.
    regions : np.ndarray
        (V,) int32 region label per vertex. Labels are integers indexing
        `region_names` below. If unknown, all zeros.
    region_names : List[str]
        Mapping from region index -> region name.
    mirror_idx : np.ndarray
        (V,) int32 index of the mirrored vertex (left-right). If no exact
        mirror found, the nearest vertex is used.
    mouth_cavity_faces : np.ndarray
        (M, 3) int32 faces that belong to the mouth cavity (inner mouth).
    metadata : dict
        Any additional metadata (source path, applied subdivision, etc.)
    """
    verts: np.ndarray
    faces: np.ndarray
    uvs: np.ndarray
    regions: np.ndarray
    region_names: List[str]
    mirror_idx: np.ndarray
    mouth_cavity_faces: np.ndarray
    metadata: Dict = None

    def save(self, path: Path) -> None:
        """
        Save the template to a compressed .npz file (numpy) with a small pickle
        for region_names and metadata.
        """
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            str(path),
            verts=self.verts.astype(np.float32),
            faces=self.faces.astype(np.int32),
            uvs=self.uvs.astype(np.float32),
            regions=self.regions.astype(np.int32),
            mirror_idx=self.mirror_idx.astype(np.int32),
            mouth_cavity_faces=self.mouth_cavity_faces.astype(np.int32),
        )
        # Save region_names and metadata as pickle alongside .npz
        pkl = path.with_suffix(path.suffix + ".meta.pkl")
        with open(pkl, "wb") as f:
            pickle.dump({"region_names": self.region_names, "metadata": self.metadata or {}}, f)

    @staticmethod
    def load(path: Path) -> "MeshTemplate":
        """
        Load a template saved with `save()`.
        """
        path = Path(path)
        data = np.load(str(path), allow_pickle=False)
        pkl = path.with_suffix(path.suffix + ".meta.pkl")
        if pkl.exists():
            with open(pkl, "rb") as f:
                meta = pickle.load(f)
            region_names = meta.get("region_names", [])
            metadata = meta.get("metadata", {})
        else:
            region_names = []
            metadata = {}
        return MeshTemplate(
            verts=data["verts"].astype(np.float32),
            faces=data["faces"].astype(np.int32),
            uvs=data["uvs"].astype(np.float32),
            regions=data["regions"].astype(np.int32),
            region_names=region_names,
            mirror_idx=data["mirror_idx"].astype(np.int32),
            mouth_cavity_faces=data["mouth_cavity_faces"].astype(np.int32),
            metadata=metadata,
        )


# ---------------------------
# Utilities
# ---------------------------
def download_if_needed(url: str, dest: Path, force: bool = False) -> Path:
    """
    Download a file from `url` to `dest` if it does not exist or if force=True.
    Returns the destination path.
    """
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not force:
        return dest
    with urllib.request.urlopen(url) as resp:
        data = resp.read()
    with open(dest, "wb") as f:
        f.write(data)
    return dest


def _parse_obj_groups(obj_path: Path) -> Tuple[List[str], Dict[str, List[int]]]:
    """
    Parse an OBJ file and return a list of group names in order and a mapping
    group_name -> list of face indices (0-based). This is a lightweight parser
    that only reads 'g' and 'f' lines to preserve grouping information.
    """
    groups: Dict[str, List[int]] = {}
    group_order: List[str] = []
    current_group = "default"
    face_idx = 0
    with open(obj_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("g ") or line.startswith("o ") or line.startswith("usemtl "):
                # group/object/usemtl line
                parts = line.split(maxsplit=1)
                if len(parts) > 1:
                    current_group = parts[1].strip()
                else:
                    current_group = "default"
                if current_group not in groups:
                    groups[current_group] = []
                    group_order.append(current_group)
            elif line.startswith("f "):
                # face line
                groups.setdefault(current_group, []).append(face_idx)
                face_idx += 1
    return group_order, groups


# ---------------------------
# Midpoint subdivision
# ---------------------------
def midpoint_subdivide(verts: np.ndarray, faces: np.ndarray, uvs: Optional[np.ndarray] = None,
                       regions_per_face: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Perform one iteration of midpoint subdivision (each triangle -> 4 triangles).
    - verts: (V,3)
    - faces: (F,3)
    - uvs: (V,2) or None
    - regions_per_face: (F,) int region index per face (optional)

    Returns:
    - new_verts: (V_new, 3)
    - new_faces: (F_new, 3)
    - new_uvs: (V_new, 2) or None
    - new_regions_per_face: (F_new,) or None
    """
    verts = np.asarray(verts, dtype=np.float64)
    faces = np.asarray(faces, dtype=np.int64)
    V = verts.shape[0]

    # Edge to midpoint index
    edge_map: Dict[Tuple[int, int], int] = {}
    midpoints: List[np.ndarray] = []
    mid_uvs: List[np.ndarray] = []

    def edge_key(a: int, b: int) -> Tuple[int, int]:
        return (a, b) if a < b else (b, a)

    # Precompute face-wise regions if provided
    has_uv = uvs is not None
    if has_uv:
        uvs = np.asarray(uvs, dtype=np.float64)

    new_faces_list: List[Tuple[int, int, int]] = []
    new_regions_list: List[int] = []

    # Start with original vertices
    new_verts = verts.tolist()
    if has_uv:
        new_uvs = uvs.tolist()
    else:
        new_uvs = None

    for fi, f in enumerate(faces):
        a, b, c = int(f[0]), int(f[1]), int(f[2])
        # For each edge, create or reuse midpoint
        mids = []
        for (i0, i1) in ((a, b), (b, c), (c, a)):
            k = edge_key(i0, i1)
            if k in edge_map:
                mids.append(edge_map[k])
            else:
                p0, p1 = verts[i0], verts[i1]
                mid = 0.5 * (p0 + p1)
                idx = len(new_verts)
                new_verts.append(mid)
                edge_map[k] = idx
                mids.append(idx)
                if has_uv:
                    uv0, uv1 = uvs[i0], uvs[i1]
                    new_uvs.append(0.5 * (uv0 + uv1))
        m_ab, m_bc, m_ca = mids

        # Create 4 new faces
        new_faces_list.append((a, m_ab, m_ca))
        new_faces_list.append((m_ab, b, m_bc))
        new_faces_list.append((m_ca, m_bc, c))
        new_faces_list.append((m_ab, m_bc, m_ca))

        # propagate region label per face if provided
        if regions_per_face is not None:
            r = int(regions_per_face[fi])
            new_regions_list.extend([r, r, r, r])

    new_verts = np.asarray(new_verts, dtype=np.float32)
    new_faces = np.asarray(new_faces_list, dtype=np.int32)
    new_uvs_arr = np.asarray(new_uvs, dtype=np.float32) if has_uv else None
    new_regions_arr = np.asarray(new_regions_list, dtype=np.int32) if regions_per_face is not None else None

    return new_verts, new_faces, new_uvs_arr, new_regions_arr


# ---------------------------
# Mirror map builder
# ---------------------------
def build_mirror_map(verts: np.ndarray, tol: float = 1e-4) -> np.ndarray:
    """
    Build a left-right mirror index map by finding, for each vertex v=(x,y,z),
    the nearest vertex to (-x, y, z) within a tolerance. If no vertex is within
    tol, the nearest vertex is used (still returns an index).
    """
    verts = np.asarray(verts, dtype=np.float64)
    V = verts.shape[0]
    mirrored = verts.copy()
    mirrored[:, 0] *= -1.0  # mirror X

    tree = cKDTree(verts)
    dists, idxs = tree.query(mirrored, k=1)
    # If distance > tol, still accept but warn (no printing here; caller may inspect)
    return np.asarray(idxs, dtype=np.int32)


# ---------------------------
# Region extraction helpers
# ---------------------------
_DEFAULT_REGION_NAMES = [
    "skin",
    "upper_lip",
    "lower_lip",
    "mouth_cavity",
    "left_eye",
    "right_eye",
    "left_brow",
    "right_brow",
    "nose_other",
]


def _assign_vertex_regions_from_face_groups(faces: np.ndarray, face_groups: Dict[str, List[int]],
                                           region_names_map: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """
    Given faces (F,3) and a mapping group_name -> list of face indices (0-based),
    produce a per-vertex region index array and a list of region names.

    region_names_map maps group_name substrings to canonical region names (e.g. 'upper_lip' -> 'upper_lip').

    Strategy:
    - Build face->region mapping by checking group names for keywords.
    - Then propagate to vertices by majority of incident faces.
    """
    F = faces.shape[0]
    face_region = np.full((F,), -1, dtype=np.int32)
    # Build canonical region name set
    canonical_names = []
    # map canonical name -> index
    cname_to_idx: Dict[str, int] = {}

    # First pass: assign face regions based on group names
    for gname, face_idxs in face_groups.items():
        # find a matching canonical region
        matched = None
        g_low = gname.lower()
        for key, canonical in region_names_map.items():
            if key in g_low:
                matched = canonical
                break
        if matched is None:
            # try to match any of the default names
            for cand in _DEFAULT_REGION_NAMES:
                if cand.replace("_", "") in g_low or cand in g_low:
                    matched = cand
                    break
        if matched is None:
            matched = "skin"  # fallback

        if matched not in cname_to_idx:
            cname_to_idx[matched] = len(canonical_names)
            canonical_names.append(matched)
        ridx = cname_to_idx[matched]
        for fi in face_idxs:
            if 0 <= fi < F:
                face_region[fi] = ridx

    # Any unassigned faces -> skin
    if -1 in face_region:
        skin_idx = cname_to_idx.get("skin")
        if skin_idx is None:
            skin_idx = len(canonical_names)
            canonical_names.append("skin")
            cname_to_idx["skin"] = skin_idx
        face_region[face_region == -1] = skin_idx

    # Propagate to vertices by majority vote of incident faces
    V = int(faces.max()) + 1
    vert_face_lists: List[List[int]] = [[] for _ in range(V)]
    for fi, f in enumerate(faces):
        for v in f:
            vert_face_lists[int(v)].append(fi)

    vert_regions = np.zeros((V,), dtype=np.int32)
    for vi, flist in enumerate(vert_face_lists):
        if not flist:
            vert_regions[vi] = cname_to_idx.get("skin", 0)
            continue
        regs = face_region[flist]
        # majority
        vals, counts = np.unique(regs, return_counts=True)
        vert_regions[vi] = int(vals[np.argmax(counts)])

    return vert_regions, canonical_names


# ---------------------------
# Mouth cavity detection
# ---------------------------
def detect_mouth_cavity_faces(faces: np.ndarray, face_groups: Dict[str, List[int]]) -> np.ndarray:
    """
    Return an array of face indices that correspond to the mouth cavity.
    Prefer explicit groups named 'mouth', 'mouth_cavity', 'inner_mouth', 'inner_lip'.
    Otherwise return an empty array (caller can attempt heuristics).
    """
    candidates = []
    for gname, face_idxs in face_groups.items():
        gl = gname.lower()
        if any(k in gl for k in ("mouth", "inner", "oral", "cavity", "inner_lip")):
            candidates.extend(face_idxs)
    if not candidates:
        return np.zeros((0, 3), dtype=np.int32)
    # Return faces as indices array
    return np.asarray(sorted(set(candidates)), dtype=np.int32)


# ---------------------------
# Main loader / builder
# ---------------------------
def load_canonical_obj(obj_path: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Dict[str, List[int]]]:
    """
    Load an OBJ using trimesh but also parse group->face mapping.

    Returns:
    - verts (V,3)
    - faces (F,3)
    - uvs (V,2) if available else None
    - face_groups: mapping group_name -> list of face indices (0-based)
    """
    obj_path = Path(obj_path)
    # Use trimesh to load geometry (it will triangulate if needed)
    mesh = trimesh.load_mesh(str(obj_path), process=False)
    if not isinstance(mesh, trimesh.Trimesh):
        # If the OBJ contains multiple meshes, merge them
        mesh = trimesh.util.concatenate(mesh.dump())  # type: ignore

    verts = np.asarray(mesh.vertices, dtype=np.float32)
    faces = np.asarray(mesh.faces, dtype=np.int32)

    # Attempt to extract per-vertex UVs. trimesh stores visual.uv per vertex or per face
    uvs = None
    try:
        if hasattr(mesh.visual, "uv") and mesh.visual.uv is not None:
            # trimesh.visual.uv is per-vertex or per-face-vertex depending on loader
            uv = np.asarray(mesh.visual.uv, dtype=np.float32)
            # If uv length equals number of vertices, good
            if uv.shape[0] == verts.shape[0] and uv.shape[1] >= 2:
                uvs = uv[:, :2]
            else:
                # If uv is per-face-vertex, we need to convert to per-vertex by averaging
                # Build per-vertex uv by averaging incident uv entries
                V = verts.shape[0]
                accum = np.zeros((V, 2), dtype=np.float64)
                counts = np.zeros((V,), dtype=np.int32)
                # trimesh stores uv per face-vertex in mesh.visual.uv.reshape((-1,2))
                # but mapping is not trivial; fallback: set None
                uvs = None
    except Exception:
        uvs = None

    # Parse groups from OBJ file to preserve region names
    group_order, face_groups = _parse_obj_groups(obj_path)

    return verts, faces, uvs, face_groups


def build_template_from_obj(obj_path: Path,
                            download_url: Optional[str] = None,
                            desired_frac: float = 0.75,
                            max_zoom_out: float = 1.6) -> MeshTemplate:
    """
    Load the canonical OBJ, perform one midpoint subdivision, propagate UVs and
    region labels, build mirror map, and detect mouth cavity faces.

    Parameters
    ----------
    obj_path : Path
        Path to canonical_face_model.obj. If it does not exist and download_url
        is provided, the file will be downloaded.
    download_url : Optional[str]
        URL to download the OBJ if missing.
    desired_frac, max_zoom_out : (unused here)
        Kept for API compatibility with Stage 0 pipeline (alignment uses these).
    """
    obj_path = Path(obj_path)
    if not obj_path.exists():
        if download_url:
            download_if_needed(download_url, obj_path)
        else:
            raise FileNotFoundError(f"OBJ not found: {obj_path}")

    verts, faces, uvs, face_groups = load_canonical_obj(obj_path)

    # Build a mapping from group names to canonical region names (heuristic)
    # Keys are substrings to match in group names -> canonical region name
    region_names_map = {
        "upperlip": "upper_lip",
        "upper_lip": "upper_lip",
        "lowerlip": "lower_lip",
        "lower_lip": "lower_lip",
        "innerlip": "mouth_cavity",
        "inner_lip": "mouth_cavity",
        "mouth": "mouth_cavity",
        "left_eye": "left_eye",
        "right_eye": "right_eye",
        "eye": "left_eye",  # fallback; will be resolved by majority
        "brow": "left_brow",
        "nose": "nose_other",
        "skin": "skin",
        "cheek": "skin",
        "lip": "lower_lip",
    }

    # Assign face->region and then vertex->region
    face_region_arr = None
    try:
        vert_regions, region_names = _assign_vertex_regions_from_face_groups(faces, face_groups, region_names_map)
    except Exception:
        # Fallback: all skin
        V = verts.shape[0]
        vert_regions = np.zeros((V,), dtype=np.int32)
        region_names = ["skin"]

    # Perform one midpoint subdivision and propagate per-face region to new faces
    # We need regions per face to propagate; build face->region by majority of vertices
    F = faces.shape[0]
    face_regions = np.zeros((F,), dtype=np.int32)
    for fi, f in enumerate(faces):
        regs = vert_regions[f]
        vals, counts = np.unique(regs, return_counts=True)
        face_regions[fi] = int(vals[np.argmax(counts)])

    new_verts, new_faces, new_uvs, new_face_regions = midpoint_subdivide(verts, faces, uvs, face_regions)

    # Convert face-region to vertex-region by majority vote
    V_new = new_verts.shape[0]
    vert_face_lists: List[List[int]] = [[] for _ in range(V_new)]
    for fi, f in enumerate(new_faces):
        for v in f:
            vert_face_lists[int(v)].append(fi)
    new_vert_regions = np.zeros((V_new,), dtype=np.int32)
    for vi, flist in enumerate(vert_face_lists):
        if not flist:
            new_vert_regions[vi] = 0
            continue
        regs = new_face_regions[flist]
        vals, counts = np.unique(regs, return_counts=True)
        new_vert_regions[vi] = int(vals[np.argmax(counts)])

    # Build mirror map
    mirror_idx = build_mirror_map(new_verts, tol=1e-3)

    # Detect mouth cavity faces (indices into new_faces)
    # Map original face indices to new face indices: each original face produced 4 faces in order
    # original fi -> new indices [4*fi, 4*fi+1, 4*fi+2, 4*fi+3]
    mouth_face_idxs_orig = detect_mouth_cavity_faces(faces, face_groups)
    if mouth_face_idxs_orig.size > 0:
        mouth_new_idxs = []
        for fi in mouth_face_idxs_orig:
            base = int(fi) * 4
            mouth_new_idxs.extend([base, base + 1, base + 2, base + 3])
        mouth_new_idxs = np.asarray(sorted(set(mouth_new_idxs)), dtype=np.int32)
    else:
        # fallback: find faces whose vertex region == mouth_cavity region index (if present)
        mouth_idx = None
        for i, name in enumerate(region_names):
            if "mouth" in name or "inner" in name:
                mouth_idx = i
                break
        if mouth_idx is not None:
            mask = new_face_regions == mouth_idx
            mouth_new_idxs = np.where(mask)[0].astype(np.int32)
        else:
            mouth_new_idxs = np.zeros((0,), dtype=np.int32)

    # Build per-vertex UVs: midpoint_subdivide returned per-vertex uvs if original had uvs
    if new_uvs is None:
        new_uvs = np.zeros((new_verts.shape[0], 2), dtype=np.float32)

    # Package metadata
    metadata = {
        "source_obj": str(obj_path),
        "subdivision": "midpoint_1x",
        "original_verts": int(verts.shape[0]),
        "original_faces": int(faces.shape[0]),
    }

    template = MeshTemplate(
        verts=new_verts.astype(np.float32),
        faces=new_faces.astype(np.int32),
        uvs=new_uvs.astype(np.float32),
        regions=new_vert_regions.astype(np.int32),
        region_names=region_names,
        mirror_idx=mirror_idx.astype(np.int32),
        mouth_cavity_faces=new_faces[mouth_new_idxs] if mouth_new_idxs.size > 0 else np.zeros((0, 3), dtype=np.int32),
        metadata=metadata,
    )
    return template


# ---------------------------
# Visualization helper
# ---------------------------
def visualize_template(template: MeshTemplate, figsize: Tuple[int, int] = (8, 8), elev: float = 20, azim: float = -60) -> None:
    """
    Simple 3D visualization of the mesh colored by region. Requires matplotlib.

    Parameters
    ----------
    template : MeshTemplate
    figsize : tuple
    elev, azim : view angles
    """
    if not _HAS_MPL:
        raise RuntimeError("matplotlib is required for visualization (install matplotlib).")

    verts = template.verts
    faces = template.faces
    regions = template.regions
    region_names = template.region_names

    # Build a color map for regions
    n_regions = max(1, (max(regions) + 1) if regions.size else 1)
    cmap = plt.get_cmap("tab10")
    colors = np.array([cmap(i % 10) for i in range(n_regions)])[:, :3]

    face_colors = colors[regions[faces].mean(axis=1).astype(int) % n_regions]

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=faces, facecolors=face_colors, linewidth=0.2, antialiased=True, shade=False)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    # Legend
    for i, name in enumerate(region_names):
        ax.plot([], [], color=colors[i], label=name)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
    plt.tight_layout()
    plt.show()


# ---------------------------
# Convenience loader
# ---------------------------
def load_or_build_template(obj_path: Optional[Path] = None,
                           download_url: Optional[str] = None,
                           cache_path: Optional[Path] = None,
                           force_rebuild: bool = False) -> MeshTemplate:
    """
    Convenience function:
    - If cache_path (.npz) exists and not force_rebuild, load it.
    - Otherwise build from obj_path (downloading if needed) and save to cache_path.

    Returns MeshTemplate.
    """
    if cache_path is None:
        cache_path = Path("data/models/canonical_face_template.npz")
    else:
        cache_path = Path(cache_path)

    if cache_path.exists() and not force_rebuild:
        try:
            return MeshTemplate.load(cache_path)
        except Exception:
            # fall through to rebuild
            pass

    if obj_path is None:
        raise ValueError("obj_path must be provided to build the template if cache is missing.")

    template = build_template_from_obj(obj_path, download_url=download_url)
    template.save(cache_path)
    return template


# ---------------------------
# Module test / example (not executed on import)
# ---------------------------
if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Build and inspect canonical face template.")
    parser.add_argument("--obj", type=Path, required=True, help="Path to canonical_face_model.obj")
    parser.add_argument("--out", type=Path, default=Path("data/models/canonical_face_template.npz"), help="Output .npz path")
    parser.add_argument("--visualize", action="store_true", help="Show a quick visualization")
    args = parser.parse_args()

    tpl = load_or_build_template(obj_path=args.obj, cache_path=args.out, force_rebuild=True)
    print("Template built:")
    print(" verts:", tpl.verts.shape)
    print(" faces:", tpl.faces.shape)
    print(" uvs:", tpl.uvs.shape)
    print(" regions:", tpl.regions.shape, "region names:", tpl.region_names)
    print(" mirror map:", tpl.mirror_idx.shape)
    print(" mouth cavity faces:", tpl.mouth_cavity_faces.shape)
    if args.visualize:
        visualize_template(tpl)


Writing template.py


In [19]:
!pip install trimesh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.0/750.0 kB 11.8 MB/s eta 0:00:00a 0:00:01


In [21]:
!curl -L -o /content/drive/MyDrive/canonical_face_model.obj https://raw.githubusercontent.com/google-ai-edge/mediapipe/refs/heads/master/mediapipe/modules/face_geometry/data/canonical_face_model.obj


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 45999  100 45999    0     0   262k      0 --:--:-- --:--:-- --:--:--  262k


In [27]:
!curl -L -o /content/drive/MyDrive/canonical_face_model.obj https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3670k  100 3670k    0     0  21.0M      0 --:--:-- --:--:-- --:--:-- 21.0M


In [23]:
!python template.py --obj /content/drive/MyDrive/canonical_face_model.obj --visualize --out /content/drive/MyDrive/output_canonical_face_template.npz

Template built:
 verts: (1833, 3)
 faces: (3592, 3)
 uvs: (1833, 2)
 regions: (1833,) region names: ['skin']
 mirror map: (1833,)
 mouth cavity faces: (0, 3)
Figure(800x800)


In [24]:
%%writefile tracker.py
"""
geometry/tracker.py

Lightweight wrapper around MediaPipe FaceLandmarker (mediapipe.tasks.vision.FaceLandmarker)
for GALNP Stage 0.

Features
- Load the official `face_landmarker_v2_with_blendshapes.task` model (download if missing).
- Provide a simple, typed API to run detection on a single image or a batch of images.
- Return a clean dictionary per image:
    {
      "landmarks": (478, 3) float32,
      "blendshapes": (52,) float32,
      "pose": (4, 4) float32,
      "confidence": float,
      "success": bool
    }
- Convenience function to also return the 5 stable alignment points (pixel coords).
- Graceful handling of failures (returns success=False and sensible defaults).
- No dependency on the rest of the GALNP codebase.
- Works with CPU or GPU (MediaPipe chooses runtime based on environment).

Dependencies
------------
- numpy
- mediapipe (mediapipe.tasks)
- tqdm is NOT required here

Usage
-----
from geometry.tracker import FaceLandmarkerWrapper, FIVE_POINT_IDX

with FaceLandmarkerWrapper() as fl:
    result = fl.detect_single(rgb_image)  # rgb_image: np.ndarray HxWx3 uint8
    # or
    results = fl.detect_batch([rgb1, rgb2])
    # get 5-point pixel coordinates:
    pts5 = fl.landmarks_to_5pt(result["landmarks"], width, height)
"""

from __future__ import annotations

import logging
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np

# Import MediaPipe Tasks API
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

LOGGER = logging.getLogger("geometry.tracker")

# Default model URL and path (official MediaPipe face_landmarker with blendshapes)
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
    "face_landmarker_v2_with_blendshapes/float16/1/face_landmarker_v2_with_blendshapes.task"
)
DEFAULT_MODEL_PATH = Path("data/models/face_landmarker_v2_with_blendshapes.task")

# Constants
NUM_LANDMARKS = 478
NUM_BLENDSHAPES = 52

# 5 stable landmark indices used for alignment (MediaPipe indexing)
FIVE_POINT_IDX: Tuple[int, ...] = (33, 263, 1, 61, 291)


# -------------------------
# Utilities
# -------------------------
def ensure_model(model_path: Path = DEFAULT_MODEL_PATH, url: str = MODEL_URL) -> Path:
    """
    Ensure the FaceLandmarker .task model exists on disk. Download if missing.

    Returns the path to the model file.
    """
    model_path = Path(model_path)
    if model_path.exists():
        return model_path
    model_path.parent.mkdir(parents=True, exist_ok=True)
    LOGGER.info("Downloading FaceLandmarker model to %s ...", model_path)
    try:
        urllib.request.urlretrieve(url, model_path)
    except Exception as exc:  # pragma: no cover - network
        if model_path.exists():
            model_path.unlink()
        raise RuntimeError(f"Could not download the model from {url}. Error: {exc}") from exc
    return model_path


def compute_inframe_fraction(landmarks_norm: np.ndarray) -> float:
    """
    Compute the fraction of landmarks whose x,y are inside [0,1].
    landmarks_norm: (N,3) with x,y normalized in [0,1] (MediaPipe convention).
    """
    if landmarks_norm is None or landmarks_norm.size == 0:
        return 0.0
    xy = landmarks_norm[:, :2]
    inside = (xy[:, 0] >= 0.0) & (xy[:, 0] <= 1.0) & (xy[:, 1] >= 0.0) & (xy[:, 1] <= 1.0)
    return float(np.mean(inside))


def landmarks_to_pixel_coords(landmarks_norm: np.ndarray, width: int, height: int) -> np.ndarray:
    """
    Convert normalized landmarks (x,y in [0,1]) to pixel coordinates (width, height).
    Returns (N,2) float64.
    """
    if landmarks_norm is None or landmarks_norm.size == 0:
        return np.zeros((0, 2), dtype=np.float64)
    pts = landmarks_norm[:, :2].astype(np.float64)
    pts[:, 0] *= width
    pts[:, 1] *= height
    return pts


# -------------------------
# Wrapper class
# -------------------------
@dataclass
class FaceLandmarkerWrapper:
    """
    Wrapper around mediapipe.tasks.vision.FaceLandmarker.

    Example
    -------
    with FaceLandmarkerWrapper(model_path=Path("...")) as fl:
        out = fl.detect_single(rgb_image)
    """

    model_path: Path = DEFAULT_MODEL_PATH
    min_confidence: float = 0.5
    num_faces: int = 1

    # Internal (initialized in __enter__)
    _landmarker: Optional[vision.FaceLandmarker] = None

    def __post_init__(self) -> None:
        # Do not auto-load model here; load lazily in __enter__ or on first call
        self.model_path = Path(self.model_path)

    def _ensure_loaded(self) -> None:
        if self._landmarker is not None:
            return
        model_file = ensure_model(self.model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(model_file)),
            running_mode=vision.RunningMode.IMAGE,
            num_faces=self.num_faces,
            min_face_detection_confidence=self.min_confidence,
            min_face_presence_confidence=self.min_confidence,
            output_face_blendshapes=True,
            output_facial_transformation_matrixes=True,
        )
        self._landmarker = vision.FaceLandmarker.create_from_options(options)
        LOGGER.debug("FaceLandmarker loaded from %s", model_file)

    def close(self) -> None:
        """Close the underlying MediaPipe landmarker if loaded."""
        if self._landmarker is not None:
            try:
                self._landmarker.close()
            except Exception:
                pass
            self._landmarker = None

    # Context manager support
    def __enter__(self) -> "FaceLandmarkerWrapper":
        self._ensure_loaded()
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self.close()

    # -------------------------
    # Detection API
    # -------------------------
    def _process_mp_result(self, result) -> Dict:
        """
        Convert a mediapipe FaceLandmarker result (single face) into the clean dict.
        If result is None or missing fields, returns success=False.
        """
        if result is None:
            return {
                "landmarks": np.zeros((NUM_LANDMARKS, 3), dtype=np.float32),
                "blendshapes": np.zeros((NUM_BLENDSHAPES,), dtype=np.float32),
                "pose": np.zeros((4, 4), dtype=np.float32),
                "confidence": 0.0,
                "success": False,
            }

        # face_landmarks is a list; we use the first face
        if not result.face_landmarks:
            return {
                "landmarks": np.zeros((NUM_LANDMARKS, 3), dtype=np.float32),
                "blendshapes": np.zeros((NUM_BLENDSHAPES,), dtype=np.float32),
                "pose": np.zeros((4, 4), dtype=np.float32),
                "confidence": 0.0,
                "success": False,
            }

        lm = result.face_landmarks[0]
        if len(lm) < NUM_LANDMARKS:
            return {
                "landmarks": np.zeros((NUM_LANDMARKS, 3), dtype=np.float32),
                "blendshapes": np.zeros((NUM_BLENDSHAPES,), dtype=np.float32),
                "pose": np.zeros((4, 4), dtype=np.float32),
                "confidence": 0.0,
                "success": False,
            }

        landmarks = np.array([[p.x, p.y, p.z] for p in lm[:NUM_LANDMARKS]], dtype=np.float32)

        # Blendshapes: result.face_blendshapes is a list of lists; pick first face
        blendshapes = np.zeros((NUM_BLENDSHAPES,), dtype=np.float32)
        try:
            cats = sorted(result.face_blendshapes[0], key=lambda c: c.index)
            blendshapes = np.array([c.score for c in cats], dtype=np.float32)
        except Exception:
            # leave zeros if missing
            pass

        # Pose: facial_transformation_matrixes
        pose = np.zeros((4, 4), dtype=np.float32)
        try:
            pose = np.asarray(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(4, 4)
        except Exception:
            pass

        confidence = compute_inframe_fraction(landmarks)

        return {
            "landmarks": landmarks,
            "blendshapes": blendshapes,
            "pose": pose,
            "confidence": float(confidence),
            "success": True,
        }

    def detect_single(self, rgb: np.ndarray) -> Dict:
        """
        Run FaceLandmarker on a single RGB image (HxWx3 uint8 or float in [0,255]).
        Returns the result dictionary described above. Always returns a dict; on
        failure success=False.
        """
        if not isinstance(rgb, np.ndarray):
            raise TypeError("rgb must be a numpy.ndarray HxWx3 uint8")

        if rgb.ndim != 3 or rgb.shape[2] != 3:
            raise ValueError("rgb must have shape HxWx3")

        self._ensure_loaded()

        # Convert to MediaPipe Image
        try:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
            result = self._landmarker.detect(mp_image)
        except Exception as exc:
            LOGGER.debug("FaceLandmarker detection failed: %s", exc)
            return {
                "landmarks": np.zeros((NUM_LANDMARKS, 3), dtype=np.float32),
                "blendshapes": np.zeros((NUM_BLENDSHAPES,), dtype=np.float32),
                "pose": np.zeros((4, 4), dtype=np.float32),
                "confidence": 0.0,
                "success": False,
            }

        return self._process_mp_result(result)

    def detect_batch(self, rgbs: Sequence[np.ndarray]) -> List[Dict]:
        """
        Run FaceLandmarker on a batch of RGB images. Returns a list of result dicts
        in the same order as the input list. Each entry follows the same schema.
        """
        if not isinstance(rgbs, (list, tuple)):
            raise TypeError("rgbs must be a list or tuple of numpy arrays")

        self._ensure_loaded()
        results: List[Dict] = []
        for img in rgbs:
            try:
                res = self.detect_single(img)
            except Exception as exc:
                LOGGER.debug("Error processing image in batch: %s", exc)
                res = {
                    "landmarks": np.zeros((NUM_LANDMARKS, 3), dtype=np.float32),
                    "blendshapes": np.zeros((NUM_BLENDSHAPES,), dtype=np.float32),
                    "pose": np.zeros((4, 4), dtype=np.float32),
                    "confidence": 0.0,
                    "success": False,
                }
            results.append(res)
        return results

    # -------------------------
    # Convenience helpers
    # -------------------------
    @staticmethod
    def landmarks_to_5pt(landmarks_norm: np.ndarray, width: int, height: int) -> np.ndarray:
        """
        Given normalized landmarks (N,3), return the 5 stable alignment points
        in pixel coordinates (5,2) in the order defined by FIVE_POINT_IDX.

        If landmarks_norm is invalid, returns an empty array shape (0,2).
        """
        if landmarks_norm is None or landmarks_norm.shape[0] < max(FIVE_POINT_IDX) + 1:
            return np.zeros((0, 2), dtype=np.float64)
        pts = landmarks_norm[list(FIVE_POINT_IDX), :2].astype(np.float64)
        pts[:, 0] *= width
        pts[:, 1] *= height
        return pts

    @staticmethod
    def landmarks_to_pixel_landmarks(landmarks_norm: np.ndarray, width: int, height: int) -> np.ndarray:
        """
        Convert full normalized landmarks (N,3) to pixel coordinates (N,3),
        where x,y are in pixels and z is scaled similarly to x (approx).
        """
        if landmarks_norm is None or landmarks_norm.size == 0:
            return np.zeros((0, 3), dtype=np.float32)
        xy = landmarks_norm[:, :2].astype(np.float64) * np.array([width, height], dtype=np.float64)
        # approximate z scaling: MediaPipe z is relative depth scaled like x
        scale = float(width)
        z = landmarks_norm[:, 2].astype(np.float64) * scale
        out = np.concatenate([xy, z[:, None]], axis=1).astype(np.float32)
        return out


# -------------------------
# Convenience module-level functions
# -------------------------
def make_landmarker(model_path: Optional[Path] = None, min_confidence: float = 0.5, num_faces: int = 1) -> FaceLandmarkerWrapper:
    """
    Create a FaceLandmarkerWrapper instance. The wrapper is a context manager
    and should be closed when done (use `with` or call .close()).
    """
    if model_path is None:
        model_path = DEFAULT_MODEL_PATH
    return FaceLandmarkerWrapper(model_path=Path(model_path), min_confidence=min_confidence, num_faces=num_faces)


# -------------------------
# Example usage (not executed on import)
# -------------------------
if __name__ == "__main__":
    import argparse
    import cv2

    parser = argparse.ArgumentParser(description="Quick test for geometry.tracker FaceLandmarkerWrapper")
    parser.add_argument("--image", type=str, required=True, help="Path to an RGB image")
    parser.add_argument("--model", type=str, default=str(DEFAULT_MODEL_PATH), help="Path to face_landmarker .task")
    args = parser.parse_args()

    img = cv2.imread(args.image, cv2.IMREAD_COLOR)
    if img is None:
        raise SystemExit("Could not read image")
    # Convert BGR -> RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with make_landmarker(Path(args.model)) as fl:
        out = fl.detect_single(img_rgb)
        print("success:", out["success"])
        print("confidence:", out["confidence"])
        print("landmarks shape:", out["landmarks"].shape)
        print("blendshapes shape:", out["blendshapes"].shape)
        print("pose shape:", out["pose"].shape)
        if out["success"]:
            pts5 = fl.landmarks_to_5pt(out["landmarks"], img_rgb.shape[1], img_rgb.shape[0])
            print("5-point pixel coords:\n", pts5)


Writing tracker.py


In [29]:
!python tracker.py --image /content/drive/MyDrive/galnp/processed/crops/001_03_neutral.jpg

2026-10-01 12:59:14.778113: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-01 12:59:14.918902: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1790859561.819356   18870 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1790859561.842147   18875 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790859561.880534   18874 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback

In [30]:
%%writefile expression_basis.py

"""
geometry/expression_basis.py

Build an expression basis ΔB from neutral <-> smiling pairs produced by Stage 0
(`prepare.py`) and provide utilities to save/load and apply the basis.

Overview
--------
- Scans a directory of processed .npz files (each expected to contain at least:
  'landmarks' (478,3) and 'blendshapes' (52,) and filename pattern "<face_id>_<expression>.npz"
  where expression is 'neutral' or 'smiling').
- Groups files by face_id and keeps pairs that have both neutral and smiling.
- For each pair:
    * Align neutral -> smiling using a 2D similarity (Umeyama) computed on x,y.
    * Apply the similarity to neutral landmarks (x,y) and scale z accordingly.
    * Compute landmark delta: smiling - aligned_neutral  (shape (N,3))
    * Compute blendshape delta: smiling_blendshapes - neutral_blendshapes (shape (52,))
- Solve a ridge regression (λ = 1e-2) to find ΔB of shape (52, N, 3) such that:
      for each sample i:  sum_k ΔB[k] * b_i[k] ≈ landmark_delta_i
  Closed-form solution used: B = Y X^T (X X^T + λ I)^{-1}
  where X is (52, M) blendshape deltas, Y is (N*3, M) stacked landmark deltas.
- Optionally lift the landmark basis to a full template mesh (V_template vertices)
  by nearest-vertex mapping (simple and fast).
- Save / load ΔB as compressed .npz (or .npz + metadata pickle).
- Provide apply_expression(verts, e_delta, delta_B) to deform vertices.

Dependencies
------------
- numpy
- scipy (for KDTree)
- typing, pathlib, dataclasses, pickle, logging

This module intentionally avoids PyTorch and MediaPipe dependencies.
"""

from __future__ import annotations

import logging
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from scipy.spatial import cKDTree

LOGGER = logging.getLogger("geometry.expression_basis")
LOGGER.addHandler(logging.NullHandler())

# Constants
NUM_LANDMARKS = 478
NUM_BLENDSHAPES = 52
DEFAULT_REGULARIZATION = 1e-2


# -------------------------
# Utility math: Umeyama similarity (2D)
# -------------------------
def estimate_similarity_transform_2d(src: np.ndarray, dst: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Estimate 2D similarity transform (scale * R, translation) mapping src -> dst.
    src, dst: (N,2) arrays. Returns (A, s) where A is (2,3) affine matrix such that:
        dst ≈ (src @ A[:, :2].T) + A[:, 2]
    and s is the uniform scale.
    Implementation follows Umeyama (1991) least-squares similarity.
    """
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    assert src.shape == dst.shape and src.ndim == 2 and src.shape[1] == 2

    n = src.shape[0]
    if n == 0:
        raise ValueError("Empty point sets for similarity estimation")

    src_mean = src.mean(axis=0)
    dst_mean = dst.mean(axis=0)
    src_c = src - src_mean
    dst_c = dst - dst_mean

    cov = (dst_c.T @ src_c) / n
    U, S, VT = np.linalg.svd(cov)
    D = np.eye(2)
    if np.linalg.det(U) * np.linalg.det(VT) < 0:
        D[-1, -1] = -1.0
    R = U @ D @ VT
    var_src = (src_c ** 2).sum() / n
    scale = float((S * np.diag(D)).sum() / var_src) if var_src > 0 else 1.0
    t = dst_mean - scale * (R @ src_mean)
    A = np.hstack([scale * R, t.reshape(2, 1)])  # (2,3)
    return A, scale


def apply_similarity_to_landmarks(landmarks: np.ndarray, A: np.ndarray, width: Optional[int] = None) -> np.ndarray:
    """
    Apply 2x3 affine similarity A to landmarks.
    landmarks: (N,3) normalized coords (x,y in [0,1]) OR pixel coords depending on caller.
    If landmarks are normalized and width is provided, x,y are scaled by width/height externally.
    This function expects landmarks in normalized coords and returns transformed landmarks in same scale.
    For our use: we will apply A in pixel coordinates, so caller should convert before calling.
    """
    lm = np.asarray(landmarks, dtype=np.float64)
    if lm.ndim != 2 or lm.shape[1] < 2:
        raise ValueError("landmarks must be (N, >=2)")
    xy = lm[:, :2]
    xy_t = (xy @ A[:, :2].T) + A[:, 2]
    # z scaling: multiply original z by scale (approx)
    # Extract scale from A as sqrt(|det(A[:2,:2])|)
    scale = float(np.sqrt(abs(np.linalg.det(A[:, :2]))))
    z = lm[:, 2] * scale if lm.shape[1] > 2 else np.zeros((lm.shape[0],), dtype=np.float64)
    return np.concatenate([xy_t, z[:, None]], axis=1)


# -------------------------
# Data container for basis
# -------------------------
@dataclass
class ExpressionBasis:
    """
    Container for expression basis ΔB.

    Attributes
    ----------
    delta_B : np.ndarray
        (K, V, 3) float32 where K = 52 (blendshape dims), V = number of vertices (landmarks or template verts).
    vertex_indices : int
        Number of vertices V.
    source : str
        Description of how the basis was built (e.g., 'landmark_space' or 'lifted_to_template').
    metadata : dict
        Additional metadata (regularization, sample count, mapping info).
    """
    delta_B: np.ndarray
    vertex_count: int
    source: str
    metadata: Dict

    def save(self, path: Path) -> None:
        """
        Save the basis to a compressed .npz and a small pickle for metadata.
        """
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(str(path), delta_B=self.delta_B.astype(np.float32))
        meta_pkl = path.with_suffix(path.suffix + ".meta.pkl")
        with open(meta_pkl, "wb") as f:
            pickle.dump({"vertex_count": int(self.vertex_count), "source": self.source, "metadata": self.metadata}, f)

    @staticmethod
    def load(path: Path) -> "ExpressionBasis":
        """
        Load a basis saved with save().
        """
        path = Path(path)
        data = np.load(str(path), allow_pickle=False)
        delta_B = data["delta_B"].astype(np.float32)
        meta_pkl = path.with_suffix(path.suffix + ".meta.pkl")
        if meta_pkl.exists():
            with open(meta_pkl, "rb") as f:
                meta = pickle.load(f)
            vertex_count = int(meta.get("vertex_count", delta_B.shape[1]))
            source = meta.get("source", "unknown")
            metadata = meta.get("metadata", {})
        else:
            vertex_count = int(delta_B.shape[1])
            source = "unknown"
            metadata = {}
        return ExpressionBasis(delta_B=delta_B, vertex_count=vertex_count, source=source, metadata=metadata)


# -------------------------
# Core builder
# -------------------------
def _load_npz_landmark(npz_path: Path) -> Optional[Dict]:
    """
    Load a single .npz produced by prepare.py and extract landmarks and blendshapes.
    Returns dict with keys: 'face_id', 'expression', 'landmarks' (N,3), 'blendshapes' (52,)
    or None if file is invalid.
    """
    try:
        data = np.load(str(npz_path), allow_pickle=True)
    except Exception as exc:
        LOGGER.debug("Failed to load %s: %s", npz_path, exc)
        return None

    # Expect 'landmarks' and 'blendshapes'
    if "landmarks" not in data or "blendshapes" not in data:
        LOGGER.debug("Missing required arrays in %s", npz_path)
        return None

    landmarks = np.asarray(data["landmarks"], dtype=np.float32)
    blendshapes = np.asarray(data["blendshapes"], dtype=np.float32)

    # Parse filename: expect <face_id>_<expression>.npz
    stem = npz_path.stem
    parts = stem.split("_")
    if len(parts) < 2:
        face_id = stem
        expression = ""
    else:
        expression = parts[-1]
        face_id = "_".join(parts[:-1])

    return {"face_id": face_id, "expression": expression, "landmarks": landmarks, "blendshapes": blendshapes, "path": str(npz_path)}


def build_expression_basis_from_dir(
    processed_dir: Path,
    regularization: float = DEFAULT_REGULARIZATION,
    lift_to_template_verts: Optional[np.ndarray] = None,
    lift_method: str = "nearest",
    save_path: Optional[Path] = None,
) -> ExpressionBasis:
    """
    Build expression basis ΔB from a directory of processed .npz files.

    Parameters
    ----------
    processed_dir : Path
        Directory containing .npz files produced by prepare.py
    regularization : float
        Ridge regression lambda (default 1e-2)
    lift_to_template_verts : Optional[np.ndarray]
        If provided, an array of template vertices (V_template, 3). The computed
        landmark-space basis (K, N, 3) will be lifted to (K, V_template, 3)
        by nearest-vertex mapping.
    lift_method : str
        'nearest' currently supported.
    save_path : Optional[Path]
        If provided, save the resulting basis to this path (.npz + .meta.pkl).

    Returns
    -------
    ExpressionBasis
    """
    processed_dir = Path(processed_dir)
    files = sorted(processed_dir.glob("*.npz"))
    if not files:
        raise FileNotFoundError(f"No .npz files found in {processed_dir}")

    # Load all files
    records = []
    for p in files:
        rec = _load_npz_landmark(p)
        if rec is not None:
            records.append(rec)

    # Group by face_id
    groups: Dict[str, Dict[str, Dict]] = {}
    for r in records:
        fid = r["face_id"]
        expr = r["expression"].lower()
        groups.setdefault(fid, {})[expr] = r

    # Collect pairs
    pairs = []
    for fid, d in groups.items():
        if "neutral" in d and "smiling" in d:
            pairs.append((d["neutral"], d["smiling"]))
    if not pairs:
        raise RuntimeError("No neutral-smiling pairs found in the processed directory")

    LOGGER.info("Found %d neutral-smiling pairs", len(pairs))

    # For each pair compute aligned landmark delta and blendshape delta
    X_list = []  # blendshape deltas (52,) per sample
    Y_list = []  # landmark deltas flattened (N*3,) per sample

    for neutral_rec, smile_rec in pairs:
        lm_neu = np.asarray(neutral_rec["landmarks"], dtype=np.float64)  # (N,3)
        lm_smi = np.asarray(smile_rec["landmarks"], dtype=np.float64)
        bs_neu = np.asarray(neutral_rec["blendshapes"], dtype=np.float64)  # (52,)
        bs_smi = np.asarray(smile_rec["blendshapes"], dtype=np.float64)

        if lm_neu.shape[0] != NUM_LANDMARKS or lm_smi.shape[0] != NUM_LANDMARKS:
            LOGGER.debug("Skipping pair %s due to unexpected landmark count", neutral_rec["path"])
            continue

        # Convert normalized x,y to pixel-like coordinates for similarity estimation.
        # Use unit square scaling: multiply by 1.0 (we only need relative positions).
        src_xy = lm_neu[:, :2].copy()
        dst_xy = lm_smi[:, :2].copy()

        # Estimate similarity transform mapping neutral -> smiling
        try:
            A, scale = estimate_similarity_transform_2d(src_xy, dst_xy)
        except Exception as exc:
            LOGGER.debug("Similarity estimation failed for %s: %s", neutral_rec["path"], exc)
            continue

        # Apply transform to neutral landmarks (in same normalized coordinate space)
        lm_neu_aligned = apply_similarity_to_landmarks(lm_neu, A)  # (N,3)

        # Compute delta: smiling - aligned_neutral
        delta_lm = lm_smi - lm_neu_aligned  # (N,3)

        # Flatten to (N*3,)
        Y_list.append(delta_lm.reshape(-1))  # length N*3

        # Blendshape delta
        delta_bs = (bs_smi - bs_neu).reshape(-1)  # (52,)
        X_list.append(delta_bs)

    X = np.asarray(X_list, dtype=np.float64).T  # shape (52, M)
    Y = np.asarray(Y_list, dtype=np.float64).T  # shape (N*3, M)

    M = X.shape[1]
    if M == 0:
        raise RuntimeError("No valid pairs after filtering")

    LOGGER.info("Solving ridge regression with %d samples", M)

    # Closed-form ridge solution: B = Y X^T (X X^T + λ I)^{-1}
    Gram = X @ X.T  # (52,52)
    reg = regularization
    Gram_reg = Gram + reg * np.eye(Gram.shape[0], dtype=np.float64)
    # Solve linear system for (X X^T + λI)^{-1} via Cholesky or np.linalg.solve
    try:
        inv_term = np.linalg.inv(Gram_reg)
    except np.linalg.LinAlgError:
        inv_term = np.linalg.pinv(Gram_reg)

    B_mat = Y @ X.T @ inv_term  # shape (N*3, 52)
    N3 = B_mat.shape[0]
    N = N3 // 3
    delta_B_landmarks = B_mat.T.reshape((NUM_BLENDSHAPES, N, 3)).astype(np.float32)  # (52, N, 3)

    metadata = {
        "num_pairs": int(M),
        "regularization": float(regularization),
        "landmark_count": int(N),
    }

    source = "landmark_space"

    # Optionally lift to template verts
    if lift_to_template_verts is not None:
        tpl_verts = np.asarray(lift_to_template_verts, dtype=np.float64)
        V_tpl = tpl_verts.shape[0]
        LOGGER.info("Lifting basis from %d landmarks to %d template verts using %s mapping", N, V_tpl, lift_method)
        # Build KDTree on landmark positions (use neutral average positions across samples? We'll use the neutral positions from the first pair)
        # For a robust mapping, compute the canonical landmark positions as the mean neutral landmarks across pairs
        mean_neutral = np.zeros((N, 3), dtype=np.float64)
        count = 0
        for neutral_rec, _ in pairs:
            lm_neu = np.asarray(neutral_rec["landmarks"], dtype=np.float64)
            if lm_neu.shape[0] == N:
                mean_neutral += lm_neu
                count += 1
        if count > 0:
            mean_neutral /= float(count)
        else:
            raise RuntimeError("Cannot compute mean neutral landmarks for lifting")

        # Build KDTree on mean_neutral (use x,y,z)
        tree = cKDTree(mean_neutral)
        dists, idxs = tree.query(tpl_verts, k=1)  # idxs: (V_tpl,)
        # Create lifted delta_B: for each blendshape k, for each template vertex v, copy delta from nearest landmark idxs[v]
        delta_B_lifted = np.zeros((NUM_BLENDSHAPES, V_tpl, 3), dtype=np.float32)
        for k in range(NUM_BLENDSHAPES):
            delta_B_lifted[k] = delta_B_landmarks[k, idxs, :]
        delta_B = delta_B_lifted
        source = "lifted_to_template_nearest"
        metadata["lift_method"] = lift_method
        metadata["lift_mapping"] = {"template_vertex_to_landmark_idx": idxs.tolist()}
    else:
        delta_B = delta_B_landmarks

    basis = ExpressionBasis(delta_B=delta_B, vertex_count=int(delta_B.shape[1]), source=source, metadata=metadata)

    if save_path is not None:
        basis.save(Path(save_path))
        LOGGER.info("Saved expression basis to %s", save_path)

    return basis


# -------------------------
# Apply expression
# -------------------------
def apply_expression(verts: np.ndarray, e_delta: np.ndarray, delta_B: np.ndarray) -> np.ndarray:
    """
    Apply an expression coefficient vector e_delta (K,) to verts (V,3) using delta_B (K,V,3).

    Returns deformed verts (V,3).

    Parameters
    ----------
    verts : np.ndarray
        (V,3) float
    e_delta : np.ndarray
        (K,) float coefficients (blendshape delta magnitudes)
    delta_B : np.ndarray
        (K,V,3) float basis

    Notes
    -----
    - K must match delta_B.shape[0], V must match delta_B.shape[1].
    - This is a linear combination: verts + sum_k e_k * delta_B[k]
    """
    verts = np.asarray(verts, dtype=np.float64)
    e_delta = np.asarray(e_delta, dtype=np.float64)
    delta_B = np.asarray(delta_B, dtype=np.float64)

    if delta_B.ndim != 3:
        raise ValueError("delta_B must be (K, V, 3)")
    K, V, C = delta_B.shape
    if C != 3:
        raise ValueError("delta_B last dimension must be 3")
    if verts.shape[0] != V:
        raise ValueError(f"verts length {verts.shape[0]} does not match delta_B vertex count {V}")
    if e_delta.shape[0] != K:
        raise ValueError(f"e_delta length {e_delta.shape[0]} does not match basis K={K}")

    # Weighted sum across K
    # result = verts + (e_delta[:,None,None] * delta_B).sum(axis=0)
    weighted = (e_delta.reshape(K, 1, 1) * delta_B).sum(axis=0)
    return (verts + weighted).astype(np.float32)


# -------------------------
# Convenience CLI-like function
# -------------------------
def build_and_save_expression_basis(
    processed_dir: Path,
    out_path: Path,
    regularization: float = DEFAULT_REGULARIZATION,
    template_verts: Optional[np.ndarray] = None,
) -> ExpressionBasis:
    """
    Convenience wrapper: build basis and save to out_path (.npz + .meta.pkl).
    """
    basis = build_expression_basis_from_dir(processed_dir, regularization=regularization, lift_to_template_verts=template_verts, save_path=out_path)
    return basis


# -------------------------
# Example usage (not executed on import)
# -------------------------
if __name__ == "__main__":
    import argparse
    import sys

    parser = argparse.ArgumentParser(description="Build expression basis from processed .npz pairs")
    parser.add_argument("--processed_dir", type=Path, required=True, help="Directory with processed .npz files")
    parser.add_argument("--out", type=Path, default=Path("data/models/expression_basis.npz"), help="Output path for basis (.npz)")
    parser.add_argument("--reg", type=float, default=DEFAULT_REGULARIZATION, help="Ridge regularization lambda")
    parser.add_argument("--template_obj_verts", type=str, default="", help="Optional: path to template verts .npy (V,3) to lift basis")
    args = parser.parse_args()

    tpl_verts = None
    if args.template_obj_verts:
        try:
            tpl_verts = np.load(args.template_obj_verts)
        except Exception as exc:
            LOGGER.error("Failed to load template verts: %s", exc)
            sys.exit(1)

    basis = build_and_save_expression_basis(args.processed_dir, args.out, regularization=args.reg, template_verts=tpl_verts)
    print("Built basis:", basis.delta_B.shape, "source:", basis.source, "metadata:", basis.metadata)


Writing expression_basis.py


In [33]:
!ls /content/drive/MyDrive/galnp/processed/meta/

001_03_neutral.npz  029_08_smiling.npz	083_03_neutral.npz  120_08_smiling.npz
001_08_smiling.npz  030_03_neutral.npz	083_08_smiling.npz  121_03_neutral.npz
002_03_neutral.npz  030_08_smiling.npz	086_03_neutral.npz  121_08_smiling.npz
002_08_smiling.npz  031_03_neutral.npz	086_08_smiling.npz  122_03_neutral.npz
003_03_neutral.npz  031_08_smiling.npz	087_03_neutral.npz  122_08_smiling.npz
003_08_smiling.npz  032_03_neutral.npz	087_08_smiling.npz  123_03_neutral.npz
004_03_neutral.npz  032_08_smiling.npz	090_03_neutral.npz  123_08_smiling.npz
004_08_smiling.npz  033_03_neutral.npz	090_08_smiling.npz  124_03_neutral.npz
005_03_neutral.npz  033_08_smiling.npz	091_03_neutral.npz  124_08_smiling.npz
005_08_smiling.npz  034_03_neutral.npz	091_08_smiling.npz  125_03_neutral.npz
006_03_neutral.npz  034_08_smiling.npz	092_03_neutral.npz  125_08_smiling.npz
006_08_smiling.npz  036_03_neutral.npz	092_08_smiling.npz  126_03_neutral.npz
007_03_neutral.npz  036_08_smiling.npz	094_03_neutral.npz  126_0

In [34]:
!python expression_basis.py --processed_dir /content/drive/MyDrive/galnp/processed/meta/ --out /content/drive/MyDrive/expression/expression_basis.npz

Built basis: (52, 478, 3) source: landmark_space metadata: {'num_pairs': 102, 'regularization': 0.01, 'landmark_count': 478}


In [35]:
%%writefile mesh_model.py
"""
geometry/mesh_model.py

MeshModel for GALNP Stage 0.

Responsibilities
- Load canonical template (from geometry.template.MeshTemplate) and expression basis
  (from geometry.expression_basis.ExpressionBasis).
- Provide a torch-based deform method:
      V_out = V_A + sum_k (e_B[k] - e_A[k]) * ΔB[k]
  supporting single-instance and batched inputs.
- Return a per-vertex correspondence attribute useful for later warping:
  - If the loaded template contains UVs and metadata.target_size, returns UV pixel
    coordinates (V,2) as the projected positions in the aligned crop.
  - Otherwise returns the input vertex positions V_A (identity mapping).
- Keep internals in torch for later differentiability; conversion helpers accept numpy.

Notes
- This module prefers PyTorch. If torch is not available, it raises ImportError.
- The class is intentionally small and focused; it does not perform rendering,
  camera projection, or image warping itself.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple, Union

import numpy as np

try:
    import torch
    from torch import Tensor
except Exception as exc:  # pragma: no cover - environment dependent
    raise ImportError("geometry.mesh_model requires PyTorch (torch).") from exc

# Local imports (assumes geometry package is on PYTHONPATH)
from geometry.template import MeshTemplate, load_or_build_template  # type: ignore
from geometry.expression_basis import ExpressionBasis  # type: ignore


ArrayLike = Union[np.ndarray, Tensor]


@dataclass
class MeshModel:
    """
    MeshModel holds a template mesh and an expression basis and provides deformation.

    Typical usage
    -------------
    model = MeshModel()
    model.load_template(path_to_template_npz)   # loads MeshTemplate
    model.load_basis(path_to_basis_npz)         # loads ExpressionBasis
    V_out, corr = model.deform(V_A, e_A, e_B)

    Methods accept numpy arrays or torch tensors. Internally tensors are torch.float32
    on CPU. If you pass CUDA tensors, operations will follow the tensor device.
    """

    template: Optional[MeshTemplate] = None
    basis: Optional[ExpressionBasis] = None
    _delta_B_torch: Optional[Tensor] = None  # (K, V, 3) torch tensor
    device: torch.device = torch.device("cpu")
    dtype: torch.dtype = torch.float32

    # -------------------------
    # Loading utilities
    # -------------------------
    def load_template(self, template_path: Optional[Path] = None, download_url: Optional[str] = None) -> MeshTemplate:
        """
        Load the canonical template. If template_path is None, uses default cache path
        and will raise if missing. Returns the loaded MeshTemplate and stores it.
        """
        if template_path is None:
            tpl = load_or_build_template(obj_path=None)  # will raise if no cache; kept for API parity
        else:
            tpl = load_or_build_template(obj_path=Path(template_path), cache_path=Path(template_path).with_suffix(".npz"))
        self.template = tpl
        return tpl

    def load_template_from_object(self, template: MeshTemplate) -> MeshTemplate:
        """
        Directly set the template from an existing MeshTemplate instance.
        """
        self.template = template
        return template

    def load_basis(self, basis_path: Optional[Path] = None, basis_obj: Optional[ExpressionBasis] = None) -> ExpressionBasis:
        """
        Load the expression basis from disk (.npz saved by ExpressionBasis.save) or
        accept an ExpressionBasis instance directly.
        """
        if basis_obj is not None:
            self.basis = basis_obj
        elif basis_path is not None:
            self.basis = ExpressionBasis.load(Path(basis_path))
        else:
            raise ValueError("Either basis_path or basis_obj must be provided")

        # Convert delta_B to torch tensor and store
        delta_B = np.asarray(self.basis.delta_B, dtype=np.float32)  # (K, V, 3)
        self._delta_B_torch = torch.from_numpy(delta_B).to(device=self.device, dtype=self.dtype)
        return self.basis

    def set_device(self, device: Union[str, torch.device]) -> None:
        """
        Move internal tensors to the specified device.
        """
        self.device = torch.device(device)
        if self._delta_B_torch is not None:
            self._delta_B_torch = self._delta_B_torch.to(self.device)

    # -------------------------
    # Helpers: conversions
    # -------------------------
    def _to_tensor(self, arr: ArrayLike, dtype: Optional[torch.dtype] = None) -> Tensor:
        """
        Convert numpy array or torch tensor to torch tensor on self.device.
        """
        if isinstance(arr, torch.Tensor):
            t = arr.to(device=self.device, dtype=(dtype or self.dtype))
        else:
            t = torch.as_tensor(np.asarray(arr), dtype=(dtype or self.dtype), device=self.device)
        return t

    def _ensure_basis_loaded(self) -> None:
        if self.basis is None or self._delta_B_torch is None:
            raise RuntimeError("Expression basis not loaded. Call load_basis() first.")

    # -------------------------
    # Core deformation API
    # -------------------------
    def deform(
        self,
        V_A: ArrayLike,
        e_A: ArrayLike,
        e_B: ArrayLike,
        return_correspondence: bool = True,
    ) -> Tuple[Tensor, Optional[Tensor]]:
        """
        Deform vertices V_A from expression e_A to expression e_B.

        Parameters
        ----------
        V_A : (V,3) or (B,V,3) numpy or torch
            Source vertices in the same vertex ordering as the basis/template.
        e_A : (K,) or (B,K) numpy or torch
            Source expression coefficients.
        e_B : (K,) or (B,K) numpy or torch
            Target expression coefficients.
        return_correspondence : bool
            If True, also return per-vertex correspondence attribute (V,2) or (B,V,2).

        Returns
        -------
        V_out : torch.Tensor
            Deformed vertices, shape (B,V,3) or (V,3) depending on input.
        correspondence : Optional[torch.Tensor]
            Per-vertex projected positions useful for warping (UV pixel coords if available),
            shape (B,V,2) or (V,2). None if return_correspondence=False.
        """
        self._ensure_basis_loaded()

        # Convert inputs to tensors
        Vt = self._to_tensor(V_A)  # (V,3) or (B,V,3)
        eA_t = self._to_tensor(e_A)
        eB_t = self._to_tensor(e_B)

        # Normalize shapes: promote to batch dimension if needed
        if Vt.ndim == 2:
            Vt = Vt.unsqueeze(0)  # (1,V,3)
            squeezed_V = True
        elif Vt.ndim == 3:
            squeezed_V = False
        else:
            raise ValueError("V_A must have shape (V,3) or (B,V,3)")

        B, V, C = Vt.shape
        if C != 3:
            raise ValueError("V_A must have 3 coordinates per vertex")

        K, V_basis, Cb = self._delta_B_torch.shape
        if V_basis != V:
            raise ValueError(f"Vertex count mismatch: basis has {V_basis} verts but V_A has {V}")

        # Ensure eA/eB shapes: (B,K)
        def _expand_coeffs(e_t: Tensor) -> Tensor:
            if e_t.ndim == 1:
                e_t = e_t.unsqueeze(0)  # (1,K)
            if e_t.ndim == 2:
                if e_t.shape[0] != B and e_t.shape[0] == 1:
                    # broadcast single coeffs to batch
                    e_t = e_t.expand(B, -1)
            if e_t.shape[0] != B:
                raise ValueError("Batch size of e_A/e_B must match V_A or be 1")
            return e_t

        eA_t = _expand_coeffs(eA_t)
        eB_t = _expand_coeffs(eB_t)

        if eA_t.shape[1] != K or eB_t.shape[1] != K:
            raise ValueError(f"Coefficient length mismatch: expected K={K}")

        # Compute delta coefficients (B,K)
        delta_e = eB_t - eA_t  # (B,K)

        # Compute weighted sum: sum_k delta_e[:,k] * delta_B[k] -> (B,V,3)
        # delta_B: (K,V,3) -> reshape to (K, V*3) or use broadcasting
        # Efficient computation: (B,K) @ (K, V, 3) -> (B,V,3)
        # Use einsum
        weighted = torch.einsum("bk,kvc->bvc", delta_e.to(self.device, self.dtype), self._delta_B_torch)  # (B,V,3)

        V_out = Vt + weighted  # (B,V,3)

        # Optionally compute correspondence attribute
        corr = None
        if return_correspondence:
            corr = self._compute_correspondence(Vt)

        # Squeeze batch dim if input was single
        if squeezed_V:
            V_out = V_out.squeeze(0)  # (V,3)
            if corr is not None:
                corr = corr.squeeze(0)  # (V,2)

        return V_out, corr

    # -------------------------
    # Correspondence helper
    # -------------------------
    def _compute_correspondence(self, Vt: Tensor) -> Optional[Tensor]:
        """
        Compute per-vertex correspondence attribute for warping.

        Strategy:
        - If template has UVs and metadata.target_size, return UV pixel coords:
            uv (V,2) in [0,1] -> uv * target_size -> (V,2) pixels
          This is returned as (B,V,2) matching batch size of Vt.
        - Otherwise return the input vertex positions Vt (B,V,3) projected to 2D by dropping Z.

        Returns None only if template is missing and Vt is invalid.
        """
        if self.template is None:
            # fallback: return XY of Vt
            return Vt[..., :2].to(self.device)

        uvs = getattr(self.template, "uvs", None)
        if uvs is not None and uvs.shape[0] == self.template.verts.shape[0]:
            # Use template metadata to determine pixel size if available
            target_size = None
            try:
                target_size = int(self.template.metadata.get("target_size", 0))
            except Exception:
                target_size = 0
            if target_size and target_size > 0:
                uv_pixels = torch.from_numpy(np.asarray(uvs, dtype=np.float32)).to(device=self.device, dtype=self.dtype) * float(target_size)
            else:
                # return normalized UVs in [0,1]
                uv_pixels = torch.from_numpy(np.asarray(uvs, dtype=np.float32)).to(device=self.device, dtype=self.dtype)
            # Broadcast to batch
            B = Vt.shape[0]
            uv_pixels = uv_pixels.unsqueeze(0).expand(B, -1, -1)  # (B,V,2)
            return uv_pixels
        else:
            # No UVs: return XY of Vt
            return Vt[..., :2].to(self.device)

    # -------------------------
    # Utility: save/load model state (template path + basis path)
    # -------------------------
    def save_state(self, path: Path) -> None:
        """
        Save a small JSON-like state describing loaded template and basis sources.
        This does not serialize tensors; it stores references for later reload.
        """
        import json

        state = {
            "template_source": getattr(self.template, "metadata", {}).get("source_obj", None) if self.template else None,
            "basis_source": getattr(self.basis, "metadata", {}).get("source", None) if self.basis else None,
            "device": str(self.device),
        }
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(state, f, indent=2)

    def load_state(self, path: Path) -> None:
        """
        Load state file produced by save_state. This only restores device setting.
        """
        import json

        path = Path(path)
        with open(path, "r", encoding="utf-8") as f:
            state = json.load(f)
        dev = state.get("device", "cpu")
        self.set_device(dev)



Writing mesh_model.py


In [36]:
%%writefile rasterizer.py
"""
geometry/rasterizer.py

Thin wrapper around nvdiffrast for GALNP Stage 0 rasterization.

Features
- Rasterize triangle meshes (batched) into G-buffer style outputs at 512x512.
- Orthographic / weak-perspective camera suitable for aligned face crops.
- Return:
    - pix_to_face: (B, H, W) int32 face index per pixel (-1 for background)
    - barycentric: (B, H, W, 3) float32 barycentric coords (sum to 1 where valid)
    - depth: (B, H, W) float32 depth (camera-space z)
    - attrs: dict of interpolated per-pixel attributes (normals, uvs, regions, etc.)
- Two entry points:
    - rasterize(...) low-level wrapper around nvdiffrast.rasterize + dr.interpolate
    - rasterize_mesh(...) higher-level convenience that accepts vertices, faces, and per-vertex attributes
- Small abstraction so backend can be swapped later (PyTorch3D, custom CUDA, etc.)

Important note about licensing
------------------------------
nvdiffrast is distributed under a non-commercial license. By using this module you
agree to comply with nvdiffrast's license terms. See the nvdiffrast repository
for details: https://github.com/NVlabs/nvdiffrast

Dependencies
------------
- torch
- nvdiffrast (nvdiffrast.torch)
- numpy

This module intentionally keeps the API narrow and typed so it can be replaced
with another rasterizer backend later with minimal changes.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np
import torch

# Try to import nvdiffrast; raise a clear error if missing
try:
    import nvdiffrast.torch as dr  # type: ignore
except Exception as exc:  # pragma: no cover - environment dependent
    raise ImportError(
        "nvdiffrast (nvdiffrast.torch) is required for geometry.rasterizer. "
        "Install from https://github.com/NVlabs/nvdiffrast and ensure CUDA is available."
    ) from exc


# Default raster resolution for aligned face crops
DEFAULT_RESOLUTION = (512, 512)


@dataclass
class RasterResult:
    """
    Container for rasterization outputs.

    Attributes
    ----------
    pix_to_face : torch.LongTensor
        (B, H, W) face index per pixel, -1 for background.
    barycentric : torch.FloatTensor
        (B, H, W, 3) barycentric coordinates for the hit triangle.
    depth : torch.FloatTensor
        (B, H, W) depth values (camera-space z) for the hit triangle.
    attrs : Dict[str, torch.Tensor]
        Interpolated per-pixel attributes. Each tensor has shape (B, H, W, C_attr).
    raw_rast : torch.Tensor
        The raw rasterizer output (B, H, W, 4) returned by nvdiffrast (kept for debugging).
    """

    pix_to_face: torch.LongTensor
    barycentric: torch.FloatTensor
    depth: torch.FloatTensor
    attrs: Dict[str, torch.Tensor]
    raw_rast: torch.Tensor


# -------------------------
# Camera / transform helpers
# -------------------------
def _build_orthographic_projection(scale: float = 1.0, translate: Tuple[float, float] = (0.0, 0.0)) -> torch.Tensor:
    """
    Build a simple 4x4 orthographic projection matrix that maps model coordinates
    (assumed to be roughly in [-0.5, 0.5] or similar) into normalized device coordinates.

    The returned matrix maps (x, y, z, 1) -> (x', y', z', 1) where x',y' are in NDC [-1,1].
    We keep z unchanged (weak-perspective / orthographic).

    Parameters
    ----------
    scale : float
        Uniform scale applied to x,y before projection.
    translate : (tx, ty)
        Translation applied to x,y after scaling.

    Returns
    -------
    proj : torch.Tensor (4,4)
    """
    tx, ty = translate
    # Scale then translate, then map to NDC [-1,1] by multiplying by 2
    # We construct a 4x4 affine matrix:
    # [ s 0 0 tx ]
    # [ 0 s 0 ty ]
    # [ 0 0 1  0 ]
    # [ 0 0 0  1 ]
    proj = torch.eye(4, dtype=torch.float32)
    proj[0, 0] = float(scale)
    proj[1, 1] = float(scale)
    proj[0, 3] = float(tx)
    proj[1, 3] = float(ty)
    return proj


# -------------------------
# Low-level rasterize wrapper
# -------------------------
def rasterize(
    vertices: torch.Tensor,
    faces: torch.Tensor,
    attributes: Optional[Dict[str, torch.Tensor]] = None,
    resolution: Tuple[int, int] = DEFAULT_RESOLUTION,
    proj: Optional[torch.Tensor] = None,
    near: float = 0.0,
    far: float = 1.0,
    cull_backfaces: bool = True,
) -> RasterResult:
    """
    Low-level rasterize using nvdiffrast.

    Parameters
    ----------
    vertices : torch.Tensor
        (B, V, 3) float32 vertex positions in model space.
    faces : torch.Tensor
        (F, 3) int64 or int32 triangle indices (shared across batch).
        If faces are per-batch, pass shape (B, F, 3) and ensure dtype is int32/int64.
    attributes : dict[str, torch.Tensor], optional
        Per-vertex attributes to interpolate. Each tensor should be (B, V, C) or (V, C).
    resolution : (H, W)
        Output raster resolution.
    proj : torch.Tensor, optional
        (4,4) projection matrix in model space. If None, an identity orthographic projection is used.
    near, far : float
        Depth range (kept for compatibility; nvdiffrast returns raw z).
    cull_backfaces : bool
        Whether to cull back-facing triangles.

    Returns
    -------
    RasterResult
    """
    # Validate inputs
    if vertices.ndim != 3:
        raise ValueError("vertices must be (B, V, 3)")
    B, V, C = vertices.shape
    if C != 3:
        raise ValueError("vertices last dim must be 3")

    # faces may be (F,3) or (B,F,3)
    if faces.ndim == 2:
        F = faces.shape[0]
        faces_t = faces.to(dtype=torch.int32, device=vertices.device)
    elif faces.ndim == 3:
        if faces.shape[0] != B:
            raise ValueError("If faces are batched, faces.shape[0] must equal vertices.shape[0]")
        F = faces.shape[1]
        faces_t = faces.to(dtype=torch.int32, device=vertices.device)
    else:
        raise ValueError("faces must be (F,3) or (B,F,3)")

    # Projection: apply proj to vertices if provided
    device = vertices.device
    if proj is None:
        proj = _build_orthographic_projection().to(device)
    else:
        proj = proj.to(device)

    # Convert vertices to homogeneous and apply proj
    # vertices: (B, V, 3) -> (B, V, 4)
    ones = torch.ones((B, V, 1), dtype=vertices.dtype, device=device)
    verts_h = torch.cat([vertices, ones], dim=2)  # (B,V,4)
    # Apply projection: (B,V,4) x (4,4)^T -> (B,V,4)
    verts_proj = torch.matmul(verts_h, proj.T)  # (B,V,4)
    # Convert to NDC by dividing by w (should be 1 for affine proj)
    verts_ndc = verts_proj[..., :3]  # (B,V,3)

    # nvdiffrast expects positions in clip space (x,y,z) where x,y in [-1,1] map to screen
    # and z is depth. We pass verts_ndc directly.
    # Prepare faces: if faces are (F,3) replicate for batch
    if faces.ndim == 2:
        faces_batch = faces_t.unsqueeze(0).expand(B, -1, -1)
    else:
        faces_batch = faces_t  # already batched

    # Call nvdiffrast rasterizer
    # dr.rasterize returns (rast, rast_db) where rast is (B, H, W, 4)
    H, W = resolution
    # nvdiffrast expects vertices as (B, V, 3) and triangles as (B, F, 3)
    rast, _ = dr.rasterize(verts_ndc.contiguous(), faces_batch.contiguous(), resolution=(H, W))

    # rast: (B, H, W, 4) where:
    #  - rast[..., 0] : triangle index + 1 (0 means background)
    #  - rast[..., 1:4] : barycentric coordinates (b0, b1, b2)
    #  - rast[..., 3] : z (depth) in clip space (interpolated)
    # Note: API details may vary across nvdiffrast versions; this is the common layout.
    # Convert to convenient tensors
    # face_idx_raw: (B,H,W) int (0 = background)
    face_idx_raw = rast[..., 0].to(dtype=torch.int32)
    # Convert to -1 for background, and zero-based indices for faces
    pix_to_face = (face_idx_raw - 1).to(dtype=torch.int64)
    pix_to_face = torch.where(face_idx_raw == 0, torch.full_like(pix_to_face, -1), pix_to_face)

    bary = rast[..., 1:4].to(dtype=torch.float32)  # (B,H,W,3)
    depth = rast[..., 3].to(dtype=torch.float32)  # (B,H,W)

    # Interpolate attributes if provided
    interp_attrs: Dict[str, torch.Tensor] = {}
    if attributes:
        for name, attr in attributes.items():
            # attr may be (B,V,C) or (V,C)
            if isinstance(attr, torch.Tensor):
                attr_t = attr.to(device)
            else:
                attr_t = torch.as_tensor(np.asarray(attr), device=device)
            if attr_t.ndim == 2:
                # (V,C) -> (B,V,C)
                attr_t = attr_t.unsqueeze(0).expand(B, -1, -1)
            if attr_t.ndim != 3:
                raise ValueError(f"Attribute {name} must be (V,C) or (B,V,C)")
            # dr.interpolate expects attributes as (B, V, C) and uses rast and faces to interpolate
            # The API: dr.interpolate(attr, rast, tri) -> (B,H,W,C)
            interp = dr.interpolate(attr_t.contiguous(), rast.contiguous(), faces_batch.contiguous())
            interp_attrs[name] = interp  # (B,H,W,C)

    return RasterResult(
        pix_to_face=pix_to_face,
        barycentric=bary,
        depth=depth,
        attrs=interp_attrs,
        raw_rast=rast,
    )


# -------------------------
# High-level convenience
# -------------------------
def rasterize_mesh(
    verts: np.ndarray,
    faces: np.ndarray,
    uv: Optional[np.ndarray] = None,
    normals: Optional[np.ndarray] = None,
    regions: Optional[np.ndarray] = None,
    resolution: Tuple[int, int] = DEFAULT_RESOLUTION,
    device: Optional[torch.device] = None,
    orthographic_scale: float = 1.0,
    orthographic_translate: Tuple[float, float] = (0.0, 0.0),
) -> RasterResult:
    """
    High-level rasterize helper that accepts numpy arrays and common per-vertex attributes.

    Parameters
    ----------
    verts : (B, V, 3) or (V, 3) numpy
    faces : (F, 3) or (B, F, 3) numpy
    uv : (V, 2) numpy optional
    normals : (V, 3) numpy optional
    regions : (V,) int numpy optional (region indices)
    resolution : (H, W)
    device : torch.device or None (defaults to cpu)
    orthographic_scale : float
    orthographic_translate : (tx, ty)

    Returns
    -------
    RasterResult
    """
    if device is None:
        device = torch.device("cpu")
    # Convert inputs to torch
    def _to_torch(x, dtype=torch.float32):
        if x is None:
            return None
        t = torch.as_tensor(np.asarray(x), dtype=dtype, device=device)
        return t

    verts_t = _to_torch(verts, dtype=torch.float32)
    faces_t = _to_torch(faces, dtype=torch.int32)
    uv_t = _to_torch(uv, dtype=torch.float32)
    normals_t = _to_torch(normals, dtype=torch.float32)
    regions_t = _to_torch(regions, dtype=torch.float32) if regions is not None else None

    # Ensure batch dimension on verts
    if verts_t.ndim == 2:
        verts_t = verts_t.unsqueeze(0)  # (1,V,3)
    if faces_t.ndim == 2:
        faces_t = faces_t  # (F,3)
    elif faces_t.ndim == 3 and faces_t.shape[0] == verts_t.shape[0]:
        faces_t = faces_t
    else:
        # If faces are (F,3) but verts are batched, replicate faces for batch
        if faces_t.ndim == 2 and verts_t.ndim == 3:
            faces_t = faces_t  # rasterize() will expand faces
        else:
            raise ValueError("faces must be (F,3) or (B,F,3)")

    # Build attributes dict
    attrs: Dict[str, torch.Tensor] = {}
    if uv_t is not None:
        attrs["uv"] = uv_t
    if normals_t is not None:
        attrs["normal"] = normals_t
    if regions_t is not None:
        # store regions as float attribute for interpolation; caller can round later
        attrs["region"] = regions_t.unsqueeze(-1) if regions_t.ndim == 1 else regions_t

    # Build projection matrix
    proj = _build_orthographic_projection(scale=orthographic_scale, translate=orthographic_translate)

    # Call low-level rasterize
    result = rasterize(
        vertices=verts_t,
        faces=faces_t,
        attributes=attrs if attrs else None,
        resolution=resolution,
        proj=proj,
    )

    # Post-process region attribute to integer if present
    if "region" in result.attrs:
        # round to nearest int and cast to long
        region_interp = result.attrs["region"]  # (B,H,W,1)
        region_int = torch.round(region_interp[..., 0]).to(dtype=torch.int64)
        result.attrs["region"] = region_int  # replace with (B,H,W) int tensor

    return result


# -------------------------
# Backend abstraction note
# -------------------------
# The functions above are intentionally small and use only:
#   - dr.rasterize(...)
#   - dr.interpolate(...)
# If you later want to swap to a different backend (PyTorch3D or custom CUDA),
# replace rasterize(...) implementation and keep the RasterResult contract.
#
# Example usage:
#   result = rasterize_mesh(verts_np, faces_np, uv=uv_np, normals=norm_np)
#   pix_to_face = result.pix_to_face.cpu().numpy()
#   bary = result.barycentric.cpu().numpy()
#   depth = result.depth.cpu().numpy()
#   uv_pixels = result.attrs.get("uv")  # (B,H,W,2) or None
#
# Keep in mind that nvdiffrast returns triangle indices with 1-based indexing in the raster buffer,
# so we convert them to -1 for background and 0-based face indices for valid pixels.

# End of file


Writing rasterizer.py


In [37]:
%%writefile gbuffer.py
"""
geometry/gbuffer.py

Build a 20-channel G-buffer (512x512) from rasterizer output + original photo
for GALNP Stage 0.

Channels (indexing 1..20 in the user's spec, zero-based here):
0.  coverage mask (1)
1.  depth (normalized) (1)
2-4. normals (3)
5-12. region one-hot (8) [skin, upper_lip, lower_lip, mouth_cavity, eye_L, eye_R, brows, nose_other]
13-14. canonical UV (2)
15-16. flow / correspondence offset (corr - pixel) / 64 (2)
17.    stretch = log det J(corr) (1)
18.    confidence (1)
19.    padding / extra (1)

Also returns a warped photo W = grid_sample(original_photo, correspondence_coords).

Notes
-----
- This module uses PyTorch for tensor ops and grid_sample.
- It expects `raster_output` to be the RasterResult returned by geometry.rasterizer.rasterize/rasterize_mesh.
- If per-pixel correspondence ("corr") is not already present in raster_output.attrs,
  you may pass per-vertex projected positions `V_A_projected` (shape (V,2) or (B,V,2))
  and `faces` (F,3) so the function can interpolate per-pixel correspondence using
  raster_output.pix_to_face and raster_output.barycentric.
- Depth normalization, stretch and confidence are computed per-image with safe fallbacks.
- The function is written to be efficient and vectorized; it supports batch processing.

Return
------
A dict with keys:
- "G": torch.Tensor (B, 20, H, W) the G-buffer
- "W": torch.Tensor (B, 3, H, W) warped photo (RGB float32 in [0,1])
- "corr": torch.Tensor (B, H, W, 2) correspondence in pixel coordinates
- "components": dict of intermediate tensors (mask, depth, normals, regions, uv, flow, stretch, confidence)
"""

from __future__ import annotations

from typing import Dict, Optional, Tuple

import torch
import torch.nn.functional as F
import numpy as np


def _ensure_batch(t: torch.Tensor) -> torch.Tensor:
    """Ensure tensor has batch dim as first dimension."""
    if t.ndim == 3:
        return t.unsqueeze(0)
    return t


def _one_hot_regions(region_map: torch.Tensor, num_regions: int = 8) -> torch.Tensor:
    """
    region_map: (B, H, W) int64
    returns: (B, num_regions, H, W) float32 one-hot
    """
    B, H, W = region_map.shape
    # clamp negative (background) to 0
    region_clamped = torch.clamp(region_map, min=0)
    oh = F.one_hot(region_clamped.long(), num_classes=num_regions)  # (B,H,W,num_regions)
    oh = oh.permute(0, 3, 1, 2).to(dtype=torch.float32)  # (B,num_regions,H,W)
    return oh


def _normalize_depth(depth: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    Normalize depth to [0,1] per image using valid pixels only.
    depth: (B,H,W)
    mask: (B,H,W) boolean
    """
    B = depth.shape[0]
    out = torch.zeros_like(depth)
    for i in range(B):
        valid = mask[i]
        if valid.any():
            d = depth[i][valid]
            dmin = float(d.min())
            dmax = float(d.max())
            if dmax > dmin:
                out[i][valid] = (depth[i][valid] - dmin) / (dmax - dmin)
            else:
                out[i][valid] = 0.0
    return out


def _compute_jacobian_and_stretch(corr: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute approximate Jacobian J of the correspondence field corr(x,y) -> (u,v)
    using finite differences and return:
      - detJ (B,H,W) and stretch = log(|detJ| + eps) (B,H,W)
    corr: (B,H,W,2) pixel coords
    mask: (B,H,W) boolean valid pixels
    """
    # corr: (B,H,W,2) -> permute to (B,2,H,W)
    corr_t = corr.permute(0, 3, 1, 2)  # (B,2,H,W)
    # Compute spatial gradients using Sobel-like kernels (finite differences)
    # kernels for dx, dy
    kernel_dx = torch.tensor([[-0.5, 0.0, 0.5]], dtype=corr_t.dtype, device=corr_t.device).reshape(1, 1, 1, 3)
    kernel_dy = kernel_dx.permute(0, 1, 3, 2)
    # pad to keep same size
    pad = (1, 1, 1, 1)
    # compute gradients for each channel u and v
    du_dx = F.conv2d(F.pad(corr_t[:, 0:1], pad, mode="replicate"), kernel_dx)
    du_dy = F.conv2d(F.pad(corr_t[:, 0:1], pad, mode="replicate"), kernel_dy)
    dv_dx = F.conv2d(F.pad(corr_t[:, 1:2], pad, mode="replicate"), kernel_dx)
    dv_dy = F.conv2d(F.pad(corr_t[:, 1:2], pad, mode="replicate"), kernel_dy)
    # du_dx etc shapes: (B,1,H,W)
    # compute detJ = du_dx * dv_dy - du_dy * dv_dx
    detJ = (du_dx * dv_dy - du_dy * dv_dx).squeeze(1)  # (B,H,W)
    # mask out invalid pixels
    detJ = detJ * mask.to(detJ.dtype)
    stretch = torch.log(torch.abs(detJ) + eps)
    # For invalid pixels set stretch to 0
    stretch = stretch * mask.to(stretch.dtype)
    return detJ, stretch


def _compute_confidence(stretch: torch.Tensor, normals: Optional[torch.Tensor], mask: torch.Tensor, mouth_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    Heuristic confidence combining:
      - penalize extreme stretch (large |stretch|)
      - favor normals facing camera (normal_z)
      - penalize mouth cavity region slightly
    stretch: (B,H,W)
    normals: (B,3,H,W) or None
    mask: (B,H,W)
    mouth_mask: (B,H,W) boolean where mouth cavity is present
    Returns confidence in [0,1] (B,H,W)
    """
    B, H, W = mask.shape
    conf = torch.zeros_like(stretch)
    # stretch contribution: map stretch to [0,1] via gaussian-like falloff
    # smaller absolute stretch -> higher score
    s = stretch
    s_abs = torch.abs(s)
    # scale parameter
    sigma = 1.0
    stretch_score = torch.exp(- (s_abs ** 2) / (2 * sigma ** 2))
    conf = stretch_score

    # normal contribution
    if normals is not None:
        # normals: (B,3,H,W)
        nz = normals[:, 2:3, :, :].squeeze(1)  # (B,H,W)
        # map nz in [-1,1] to [0,1] favoring >0 (facing camera)
        normal_score = (nz + 1.0) / 2.0
        conf = conf * 0.6 + normal_score * 0.4

    # mouth penalty
    if mouth_mask is not None:
        conf = conf * (~mouth_mask).to(conf.dtype) + conf * 0.6 * mouth_mask.to(conf.dtype)

    # mask invalid pixels
    conf = conf * mask.to(conf.dtype)
    # clamp
    conf = torch.clamp(conf, 0.0, 1.0)
    return conf


def build_gbuffer(
    raster_output,
    original_image: torch.Tensor,
    V_A_projected: Optional[torch.Tensor] = None,
    faces: Optional[torch.Tensor] = None,
    resolution: Tuple[int, int] = (512, 512),
    flow_scale: float = 64.0,
) -> Dict[str, torch.Tensor]:
    """
    Build the G-buffer and warped photo.

    Parameters
    ----------
    raster_output : object
        The RasterResult returned by geometry.rasterizer.rasterize/rasterize_mesh.
        Expected attributes:
          - pix_to_face: (B,H,W) int tensor (-1 background)
          - barycentric: (B,H,W,3) float tensor
          - depth: (B,H,W) float tensor
          - attrs: dict of interpolated attributes (e.g., 'uv', 'normal', 'region', 'corr')
    original_image : torch.Tensor
        (B, 3, H, W) float32 in [0,1] or (3,H,W) -> will be expanded.
    V_A_projected : Optional[torch.Tensor]
        Per-vertex projected positions in pixel coords (V,2) or (B,V,2). Used if raster_output.attrs lacks 'corr'.
    faces : Optional[torch.Tensor]
        (F,3) int tensor required if V_A_projected is provided and raster_output.attrs lacks 'corr'.
    resolution : (H, W)
    flow_scale : float
        Divisor for flow normalization (corr - pixel) / flow_scale

    Returns
    -------
    dict with keys:
      - "G": (B,20,H,W) float32
      - "W": (B,3,H,W) warped photo
      - "corr": (B,H,W,2) correspondence in pixel coords
      - "components": dict of intermediate tensors
    """
    device = original_image.device
    H, W = resolution
    # Ensure batch dims
    pix_to_face = _ensure_batch(raster_output.pix_to_face)  # (B,H,W)
    bary = _ensure_batch(raster_output.barycentric)  # (B,H,W,3)
    depth = _ensure_batch(raster_output.depth)  # (B,H,W)
    attrs = getattr(raster_output, "attrs", {})

    B = pix_to_face.shape[0]

    # Ensure original_image shape (B,3,H,W)
    if original_image.ndim == 3:
        original_image = original_image.unsqueeze(0)
    if original_image.shape[2] != H or original_image.shape[3] != W:
        # If original image resolution differs, we will still grid_sample using corr coords in pixel space.
        # But it's recommended to pass aligned 512x512 crops.
        pass

    # coverage mask
    mask = (pix_to_face >= 0)  # (B,H,W) bool

    # depth normalized
    depth_norm = _normalize_depth(depth, mask)  # (B,H,W)

    # normals: try raster_output.attrs['normal'] expected shape (B,H,W,3) or (B,3,H,W)
    normals = None
    if "normal" in attrs:
        n = attrs["normal"]
        if n.ndim == 4 and n.shape[-1] == 3:
            normals = n.permute(0, 3, 1, 2)  # (B,3,H,W)
        elif n.ndim == 4 and n.shape[1] == 3:
            normals = n  # (B,3,H,W)
        else:
            # try to reshape
            try:
                normals = n.permute(0, 3, 1, 2)
            except Exception:
                normals = None
    # If normals missing, set default facing camera (0,0,1)
    if normals is None:
        normals = torch.zeros((B, 3, H, W), dtype=torch.float32, device=device)
        normals[:, 2, :, :] = 1.0

    # regions: expect attrs['region'] (B,H,W) int or float
    if "region" in attrs:
        region_map = attrs["region"]
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        if region_map.ndim == 3:
            region_map = region_map
        else:
            # try permute
            try:
                region_map = region_map.squeeze(-1)
            except Exception:
                region_map = torch.zeros((B, H, W), dtype=torch.int64, device=device)
    else:
        # default all skin (region 0)
        region_map = torch.zeros((B, H, W), dtype=torch.int64, device=device)

    # region one-hot (8 channels)
    region_onehot = _one_hot_regions(region_map, num_regions=8)  # (B,8,H,W)

    # canonical UVs
    if "uv" in attrs:
        uv = attrs["uv"]  # (B,H,W,2) or (B,2,H,W)
        if uv.ndim == 4 and uv.shape[-1] == 2:
            uv_pixels = uv.permute(0, 3, 1, 2)  # (B,2,H,W)
        elif uv.ndim == 4 and uv.shape[1] == 2:
            uv_pixels = uv
        else:
            uv_pixels = torch.zeros((B, 2, H, W), dtype=torch.float32, device=device)
    else:
        uv_pixels = torch.zeros((B, 2, H, W), dtype=torch.float32, device=device)

    # correspondence corr: prefer raster_output.attrs['corr'] else interpolate from V_A_projected + faces
    corr = None  # (B,H,W,2)
    if "corr" in attrs:
        c = attrs["corr"]
        if c.ndim == 4 and c.shape[-1] == 2:
            corr = c  # (B,H,W,2)
        elif c.ndim == 4 and c.shape[1] == 2:
            corr = c.permute(0, 2, 3, 1)
        else:
            corr = None

    if corr is None and V_A_projected is not None and faces is not None:
        # Interpolate per-pixel correspondence using pix_to_face and barycentric
        # V_A_projected: (V,2) or (B,V,2)
        Vproj = V_A_projected
        if isinstance(Vproj, torch.Tensor):
            Vproj_t = Vproj.to(device)
        else:
            Vproj_t = torch.as_tensor(np.asarray(Vproj), dtype=torch.float32, device=device)
        if Vproj_t.ndim == 2:
            Vproj_t = Vproj_t.unsqueeze(0).expand(B, -1, -1)  # (B,V,2)
        # faces: (F,3)
        faces_t = faces.to(device=device, dtype=torch.long)
        # Build per-pixel corr by indexing faces
        # pix_to_face: (B,H,W) with -1 for background
        # bary: (B,H,W,3)
        # We'll create corr by gathering vertex coords for each face index
        Bp, Hp, Wp = pix_to_face.shape
        corr_out = torch.zeros((B, Hp, Wp, 2), dtype=torch.float32, device=device)
        # For efficiency, vectorize: create face_idx_flat and bary_flat
        face_idx_flat = pix_to_face.reshape(Bp, -1)  # (B, HW)
        bary_flat = bary.reshape(Bp, -1, 3)  # (B, HW, 3)
        for b in range(Bp):
            face_idx_b = face_idx_flat[b]  # (HW,)
            bary_b = bary_flat[b]  # (HW,3)
            valid_mask = face_idx_b >= 0
            if valid_mask.any():
                valid_faces = face_idx_b[valid_mask]  # indices into faces
                # gather vertex indices
                face_vertices = faces_t[valid_faces]  # (N_valid, 3)
                # gather per-vertex projected coords
                v0 = Vproj_t[b, face_vertices[:, 0], :]  # (N_valid,2)
                v1 = Vproj_t[b, face_vertices[:, 1], :]
                v2 = Vproj_t[b, face_vertices[:, 2], :]
                bary_vals = bary_b[valid_mask]  # (N_valid,3)
                corr_vals = v0 * bary_vals[:, 0:1] + v1 * bary_vals[:, 1:2] + v2 * bary_vals[:, 2:3]  # (N_valid,2)
                corr_out[b].view(-1, 2)[valid_mask] = corr_vals
        corr = corr_out  # (B,H,W,2)

    # If still None, set corr to pixel centers (identity)
    if corr is None:
        # pixel coordinates: x in [0,W-1], y in [0,H-1]
        xs = torch.linspace(0.0, W - 1.0, W, device=device)
        ys = torch.linspace(0.0, H - 1.0, H, device=device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        corr = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).expand(B, -1, -1, -1).to(dtype=torch.float32)

    # flow = (corr - pixel_coords) / flow_scale
    xs = torch.linspace(0.0, W - 1.0, W, device=device)
    ys = torch.linspace(0.0, H - 1.0, H, device=device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
    pixel_coords = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).expand(B, -1, -1, -1)  # (B,H,W,2)
    flow = (corr - pixel_coords) / float(flow_scale)  # (B,H,W,2)

    # stretch: compute Jacobian det and log det
    detJ, stretch = _compute_jacobian_and_stretch(corr, mask)

    # confidence
    normals_for_conf = normals  # (B,3,H,W)
    mouth_mask = None
    # mouth_mask from region_map where region index corresponds to mouth cavity; find index if present
    # assume region mapping: 0 skin, 1 upper_lip, 2 lower_lip, 3 mouth_cavity, 4 left_eye, 5 right_eye, 6 brows, 7 nose_other
    mouth_mask = (region_map == 3)
    confidence = _compute_confidence(stretch, normals_for_conf, mask, mouth_mask)

    # Prepare channels
    # 0 coverage mask (float)
    coverage = mask.to(dtype=torch.float32).unsqueeze(1)  # (B,1,H,W)
    # 1 depth_norm (float)
    depth_ch = depth_norm.unsqueeze(1)  # (B,1,H,W)
    # 2-4 normals (B,3,H,W)
    normals_ch = normals  # already (B,3,H,W)
    # 5-12 region one-hot (B,8,H,W)
    regions_ch = region_onehot  # (B,8,H,W)
    # 13-14 canonical UV (B,2,H,W)
    uv_ch = uv_pixels  # (B,2,H,W)
    # 15-16 flow (B,2,H,W)
    flow_ch = flow.permute(0, 3, 1, 2)  # (B,2,H,W)
    # 17 stretch (B,1,H,W)
    stretch_ch = stretch.unsqueeze(1)
    # 18 confidence (B,1,H,W)
    conf_ch = confidence.unsqueeze(1)
    # 19 padding zeros
    pad_ch = torch.zeros((B, 1, H, W), dtype=torch.float32, device=device)

    # Concatenate in order to form G (B,20,H,W)
    G = torch.cat([coverage, depth_ch, normals_ch, regions_ch, uv_ch, flow_ch, stretch_ch, conf_ch, pad_ch], dim=1)
    assert G.shape[1] == 20, f"G-buffer channel mismatch: got {G.shape[1]} channels"

    # Warped photo W: grid_sample expects normalized coords in [-1,1] with (x,y) order
    # Build sampling grid from corr (pixel coords) to normalized [-1,1]
    # original_image: (B,3,H_img,W_img) may be same resolution; assume same H,W for aligned crops
    B_img = original_image.shape[0]
    if B_img != B:
        # broadcast original image if needed
        if B_img == 1:
            original_image = original_image.expand(B, -1, -1, -1)
        else:
            raise ValueError("Batch size of original_image must match raster_output or be 1")

    # Normalize corr to [-1,1] for grid_sample: x -> -1..1, y -> -1..1
    # Note grid_sample expects grid in shape (B,H,W,2) with last dim (x_norm, y_norm)
    x_norm = (corr[..., 0] / (W - 1)) * 2.0 - 1.0
    y_norm = (corr[..., 1] / (H - 1)) * 2.0 - 1.0
    grid = torch.stack([x_norm, y_norm], dim=-1)  # (B,H,W,2)
    # grid_sample expects float32
    grid = grid.to(dtype=original_image.dtype)
    # sample
    # align_corners True to match pixel mapping precisely
    W_warp = F.grid_sample(original_image, grid, mode="bilinear", padding_mode="zeros", align_corners=True)

    components = {
        "mask": mask,
        "depth": depth_norm,
        "normals": normals,
        "regions_onehot": region_onehot,
        "uv": uv_pixels,
        "flow": flow,
        "stretch": stretch,
        "confidence": confidence,
        "pixel_coords": pixel_coords,
    }

    return {
        "G": G,
        "W": W_warp,
        "corr": corr,
        "components": components,
    }


Writing gbuffer.py


In [45]:
%%writefile visualize_stage0.py
#!/usr/bin/env python3
"""
tools/visualize_stage0.py

Visualize Stage-0 outputs for one identity (neutral + smiling).

Layout (2 rows x 5 cols):
    [ neutral crop ] [ smiling crop ] [ mesh overlay ] [ depth ] [ normals ]
    [ flow         ] [ stretch      ] [ confidence   ] [ warp self ] [ warp neutral->smile ]

Usage:
    python tools/visualize_stage0.py --face-id 1001
    python tools/visualize_stage0.py 1001
    python tools/visualize_stage0.py 1001 --data-root /content/output --out /content/stage0_1001.png

The script looks for a directory under --data-root whose name contains the
face id (e.g. /content/output/1001/, /content/output/face_1001/, ...) and
then probes a set of common filenames inside it for each panel.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")  # safe in Colab / headless
import matplotlib.pyplot as plt
from matplotlib import image as mpimg


# ---------------------------------------------------------------------------
# IO helpers
# ---------------------------------------------------------------------------

def _read_image(path: Path):
    """Load an image or npy array to display."""
    if path is None or not path.exists():
        return None
    suf = path.suffix.lower()
    if suf in (".npy", ".npz"):
        arr = np.load(path, allow_pickle=True)
        if isinstance(arr, np.lib.npyio.NpzFile):
            key = list(arr.files)[0]
            arr = arr[key]
        return np.asarray(arr)
    return mpimg.imread(str(path))


def _read_array(path: Path):
    if path is None or not path.exists():
        return None
    suf = path.suffix.lower()
    if suf == ".npy":
        return np.load(path, allow_pickle=True)
    if suf == ".npz":
        with np.load(path, allow_pickle=True) as z:
            # Prefer a canonical key if present
            for k in ("arr_0", "data", "depth", "normal", "flow", "stretch", "conf"):
                if k in z.files:
                    return z[k]
            return z[z.files[0]]
    # fall back to image
    return mpimg.imread(str(path))


def _first_existing(id_dir: Path, patterns):
    """First path matching any pattern (direct glob first, then recursive)."""
    for pat in patterns:
        hits = sorted(id_dir.glob(pat))
        if hits:
            return hits[0]
    for pat in patterns:
        hits = sorted(id_dir.rglob(pat))
        if hits:
            return hits[0]
    return None


def _find_identity_dir(root: Path, face_id: str) -> Path:
    """Locate the processed directory for a face id."""
    face_id = str(face_id)
    direct = [
        root / face_id,
        root / f"face_{face_id}",
        root / f"id_{face_id}",
        root / f"identity_{face_id}",
        root / "identities" / face_id,
        root / "stage0" / face_id,
    ]
    for c in direct:
        if c.is_dir():
            return c

    # Fallback: any dir under root whose name contains the id
    matches = [p for p in root.rglob(f"*{face_id}*") if p.is_dir()]
    if matches:
        # Prefer the shortest path (most likely the identity root)
        matches.sort(key=lambda p: len(p.parts))
        return matches[0]

    # Give a helpful error listing what we did see
    listing = sorted([p.name for p in root.iterdir()]) if root.is_dir() else []
    raise FileNotFoundError(
        f"Could not find processed data for face_id={face_id} under {root}.\n"
        f"Top-level entries in {root}: {listing[:20]}{' ...' if len(listing) > 20 else ''}"
    )


# ---------------------------------------------------------------------------
# Panel resolution
# ---------------------------------------------------------------------------

def _resolve_panels(id_dir: Path) -> dict:
    """Best-effort mapping of panel key -> file path on disk."""

    # Names for the neutral / smiling subfolders (which may or may not exist)
    neutral_dirs = ["neutral", "neutral_front", "N", "A", "source", "src"]
    smiling_dirs = ["smiling", "smile", "smiling_front", "S", "B", "target", "tgt"]

    def under(dirs, patterns):
        for d in dirs:
            sub = id_dir / d
            if sub.is_dir():
                p = _first_existing(sub, patterns)
                if p is not None:
                    return p
        return None

    return {
        "neutral_crop": (
            _first_existing(id_dir, [
                "neutral*crop*.png", "neutral*crop*.jpg", "neutral*image*.png",
                "neutral.png", "neutral.jpg", "neutral/*.png", "neutral/*.jpg",
                "*neutral*crop*",
            ])
            or under(neutral_dirs, ["crop*.png", "crop*.jpg", "image*.png",
                                    "*.png", "*.jpg"])
        ),
        "smiling_crop": (
            _first_existing(id_dir, [
                "smil*crop*.png", "smil*crop*.jpg", "smil*image*.png",
                "smiling.png", "smiling.jpg", "smiling/*.png", "smiling/*.jpg",
                "*smil*crop*",
            ])
            or under(smiling_dirs, ["crop*.png", "crop*.jpg", "image*.png",
                                    "*.png", "*.jpg"])
        ),
        "mesh_overlay": _first_existing(id_dir, [
            "*mesh*overlay*.png", "*mesh*overlay*.jpg",
            "*overlay*.png", "*mesh*.png",
        ]),
        "depth": (
            _first_existing(id_dir, ["*depth*.npy", "*depth*.png"])
            or under(neutral_dirs, ["depth*.npy", "*depth*"])
        ),
        "normals": (
            _first_existing(id_dir, ["*normal*.npy", "*normal*.png"])
            or under(neutral_dirs, ["normal*.npy", "*normal*"])
        ),
        "flow": _first_existing(id_dir, [
            "*flow*.npy", "*correspond*.npy", "*flow*.png", "*correspond*.png",
        ]),
        "stretch": _first_existing(id_dir, ["*stretch*.npy", "*stretch*.png"]),
        "confidence": _first_existing(id_dir, ["*conf*.npy", "*conf*.png",
                                                "*confidence*"]),
        "warped_self": _first_existing(id_dir, [
            "*warp*self*.png", "*identity_warp*.png", "*warp_A2A*.png",
            "*warped_neutral*.png",
        ]),
        "warped_neutral2smile": _first_existing(id_dir, [
            "*neutral*smil*.png", "*warp*smil*.png", "*warped_smil*.png",
            "*A2B*.png", "*N2S*.png",
        ]),
    }


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------

def _imshow(ax, data, title):
    if data is None:
        ax.text(0.5, 0.5, "missing", ha="center", va="center",
                transform=ax.transAxes, color="crimson", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(title, fontsize=9)
        return

    data = np.asarray(data)

    if data.ndim == 2:
        ax.imshow(data, cmap="viridis")
    elif data.ndim == 3 and data.shape[-1] in (3, 4):
        if data.dtype.kind == "f":
            data = np.clip(data, 0.0, 1.0)
        ax.imshow(data)
    elif data.ndim == 3 and data.shape[0] in (3, 4):
        img = np.transpose(data, (1, 2, 0))
        if img.dtype.kind == "f":
            img = np.clip(img, 0.0, 1.0)
        ax.imshow(img)
    elif data.ndim == 3 and data.shape[-1] == 2:
        # flow field (u, v)
        u, v = data[..., 0], data[..., 1]
        mag = np.sqrt(u * u + v * v)
        ax.imshow(mag, cmap="magma")
    else:
        # squeeze any odd leading dims
        flat = data
        while flat.ndim > 3:
            flat = flat[0]
        if flat.ndim == 3 and flat.shape[0] in (1, 3, 4):
            flat = np.transpose(flat, (1, 2, 0))
        ax.imshow(flat if flat.ndim == 2 else flat[..., 0], cmap="viridis")

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=9)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

PANELS = [
    ("neutral_crop",         "Original neutral crop"),
    ("smiling_crop",         "Original smiling crop"),
    ("mesh_overlay",         "Deformed mesh overlay"),
    ("depth",                "Depth"),
    ("normals",              "Normals"),
    ("flow",                 "Correspondence / flow"),
    ("stretch",              "Stretch map"),
    ("confidence",           "Confidence map"),
    ("warped_self",          "Warped image (e_B = e_A)"),
    ("warped_neutral2smile", "Warped image (neutral \u2192 smiling)"),
]


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("face_id", nargs="?", help="face id to visualize")
    ap.add_argument("--face-id", dest="face_id_opt",
                    help="face id (flag form)")
    ap.add_argument("--data-root", default="/content/output",
                    help="root directory with processed identities (default: /content/output)")
    ap.add_argument("--out", default=None,
                    help="output PNG path (default: ./stage0_<face_id>.png)")
    ap.add_argument("--dpi", type=int, default=140)
    args = ap.parse_args(argv)

    face_id = args.face_id or args.face_id_opt
    if not face_id:
        ap.error("face_id is required (positional or --face-id)")

    data_root = Path(args.data_root)
    if not data_root.exists():
        print(f"[error] data-root does not exist: {data_root}", file=sys.stderr)
        return 2

    id_dir = _find_identity_dir(data_root, str(face_id))
    print(f"[visualize_stage0] identity dir: {id_dir}")

    panels = _resolve_panels(id_dir)

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for ax, (key, title) in zip(axes.flat, PANELS):
        path = panels.get(key)
        data = None
        if path is None:
            print(f"[warn] no file found for panel '{key}'")
        else:
            try:
                if key in ("depth", "normals", "flow", "stretch", "confidence"):
                    data = _read_array(path)
                else:
                    data = _read_image(path)
            except Exception as exc:  # noqa: BLE001
                print(f"[warn] failed to load '{key}' from {path}: {exc}")
        _imshow(ax, data, title)

    fig.suptitle(f"Stage-0 visualization  \u2014  face_id={face_id}", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])

    out = Path(args.out) if args.out else Path(f"stage0_{face_id}.png")
    fig.savefig(out, dpi=args.dpi)
    plt.close(fig)
    print(f"[visualize_stage0] wrote {out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

Overwriting visualize_stage0.py


In [63]:
!python visualize_stage0.py \
    --face_id 002 \
    --processed_dir /content/drive/MyDrive/galnp/processed/meta \
    --crop_dir      /content/drive/MyDrive/galnp/processed/crops \
    --basis         /content/drive/MyDrive/expression/expression_basis.npz \
    --out_dir       /content/visualizations/

[visualize_stage0] subject      = 002
[visualize_stage0] neutral npz  = /content/drive/MyDrive/galnp/processed/meta/002_03_neutral.npz
[visualize_stage0] smiling npz  = /content/drive/MyDrive/galnp/processed/meta/002_08_smiling.npz
[visualize_stage0] neutral img  = /content/drive/MyDrive/galnp/processed/crops/002_03_neutral.jpg
[visualize_stage0] smiling img  = /content/drive/MyDrive/galnp/processed/crops/002_08_smiling.jpg
[visualize_stage0] wrote /content/visualizations/stage0_002.png


In [59]:
!ls /content/drive/MyDrive/galnp/processed/meta | grep 001

001_03_neutral.npz
001_08_smiling.npz


In [64]:
# Copy the folder and all its contents to your main Google Drive directory
%cp -r /content/output /content/drive/MyDrive/output_galnp


## STAGE 1

In [65]:
%%writefile stage1_data.py
"""
stage1_data.py — Stage 1 same-identity pair dataset.

Per sample yields:
    A_rgb     (3,512,512) float [0,1]   — A's aligned crop
    B_rgb     (3,512,512) float [0,1]   — B's crop, realigned to A's crop frame
    W512      (3,512,512) float [0,1]   — A warped to B's mesh shape
    G128      (20,128,128) float        — G-buffer for P128
    mask128   (128,128)    float 0/1    — mesh coverage
    A_lm, B_lm (478,3) float32          — landmarks in crop pixel coords
    e_A, e_B   (52,)   float32          — blendshape scores
"""
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import torch
from scipy.spatial import Delaunay
from torch.utils.data import Dataset


# ---------------------------------------------------------------------------
# Discovery (subject-prefix match; handles 001_03_neutral ↔ 001_08_smiling)
# ---------------------------------------------------------------------------

def _norm_id(raw) -> str:
    s = str(raw).strip()
    if s.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        s = Path(s).stem
    return "_".join(str(int(p)) if p.isdigit() else p for p in s.split("_"))


def _find_crop(crop_dir: Path, stem: str, expr: str):
    for ext in (".jpg", ".jpeg", ".png"):
        c = crop_dir / f"{stem}_{expr}{ext}"
        if c.exists():
            return c
    hits = sorted(crop_dir.glob(f"{stem}_{expr}.*"))
    return hits[0] if hits else None


def discover_pairs(meta_dir: Path, crop_dir: Path) -> List[Dict]:
    meta_dir, crop_dir = Path(meta_dir), Path(crop_dir)
    pairs: List[Dict] = []
    for npath in sorted(meta_dir.glob("*_neutral.npz")):
        nstem = npath.stem[:-len("_neutral")]
        subject = nstem.split("_")[0]
        subj_norm = _norm_id(subject)
        spath = next(
            (p for p in sorted(meta_dir.glob("*_smiling.npz"))
             if _norm_id(p.stem[:-len("_smiling")].split("_")[0]) == subj_norm),
            None,
        )
        if spath is None:
            continue
        sstem = spath.stem[:-len("_smiling")]
        n_img = _find_crop(crop_dir, nstem, "neutral")
        s_img = _find_crop(crop_dir, sstem, "smiling")
        if n_img is None or s_img is None:
            continue
        pairs.append({
            "subject": subject,
            "A_stem": nstem, "B_stem": sstem,
            "A_npz": npath, "B_npz": spath,
            "A_img": n_img, "B_img": s_img,
        })
    return pairs


def _read_rgb(path, affine_2x3: np.ndarray | None = None, out_size: int = 512) -> np.ndarray:
    """Read image, optionally warp by a 2×3 affine to out_size×out_size."""
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    if affine_2x3 is not None and not np.allclose(
        affine_2x3, np.array([[1, 0, 0], [0, 1, 0]], dtype=np.float32), atol=1e-4
    ):
        bgr = cv2.warpAffine(
            bgr, affine_2x3, (out_size, out_size),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0),
        )
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


# ---------------------------------------------------------------------------
# Umeyama similarity (temporary B→A realignment; goes to ~identity once you
# switch prepare.py to the shared-alignment version)
# ---------------------------------------------------------------------------

def _umeyama_2d(src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    n = src.shape[0]
    sm, dm = src.mean(0), dst.mean(0)
    sc, dc = src - sm, dst - dm
    cov = dc.T @ sc / n
    u, s, vt = np.linalg.svd(cov)
    d = np.ones(2)
    if np.linalg.det(u) * np.linalg.det(vt) < 0:
        d[-1] = -1.0
    rot = u @ np.diag(d) @ vt
    var = (sc ** 2).sum() / n
    scale = float((s * d).sum() / max(var, 1e-9))
    t = dm - scale * rot @ sm
    return np.hstack([scale * rot, t[:, None]]).astype(np.float32)


def _apply_affine_2d(pts: np.ndarray, A: np.ndarray) -> np.ndarray:
    """Apply (2,3) affine to (N,3) points; scale z by sqrt(|det A_2x2|)."""
    xy = pts[:, :2] @ A[:, :2].T + A[:, 2]
    scale = float(np.sqrt(abs(np.linalg.det(A[:, :2]))))
    z = pts[:, 2:3] * scale
    return np.concatenate([xy, z], axis=1).astype(np.float32)


# ---------------------------------------------------------------------------
# Correspondence / warping
# ---------------------------------------------------------------------------

def delaunay_simplices(xy: np.ndarray) -> np.ndarray:
    return Delaunay(xy).simplices.astype(np.int32)


def build_sampling_grid(
    src_xy: np.ndarray, dst_xy: np.ndarray, simplices: np.ndarray,
    H: int, W: int,
) -> torch.Tensor:
    """
    F.grid_sample grid (H,W,2) mapping destination pixel -> source coord.
    Uncovered pixels get identity.
    """
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    map_x = xx.copy()
    map_y = yy.copy()

    for tri in simplices:
        d = dst_xy[tri]; s = src_xy[tri]
        x0 = max(0, int(np.floor(d[:, 0].min())))
        x1 = min(W, int(np.ceil(d[:, 0].max())) + 1)
        y0 = max(0, int(np.floor(d[:, 1].min())))
        y1 = min(H, int(np.ceil(d[:, 1].max())) + 1)
        if x1 <= x0 or y1 <= y0:
            continue
        xs, ys = np.meshgrid(np.arange(x0, x1, dtype=np.float32),
                             np.arange(y0, y1, dtype=np.float32))
        ax, ay = d[1, 0] - d[0, 0], d[1, 1] - d[0, 1]
        bx, by = d[2, 0] - d[0, 0], d[2, 1] - d[0, 1]
        den = ax * by - bx * ay
        if abs(den) < 1e-8:
            continue
        px, py = xs - d[0, 0], ys - d[0, 1]
        b1 = (px * by - py * bx) / den
        b2 = (ax * py - ay * px) / den
        b0 = 1.0 - b1 - b2
        inside = (b0 >= -1e-6) & (b1 >= -1e-6) & (b2 >= -1e-6)
        if not inside.any():
            continue
        xi = xs[inside].astype(np.int32); yi = ys[inside].astype(np.int32)
        b0i, b1i, b2i = b0[inside], b1[inside], b2[inside]
        map_x[yi, xi] = b0i * s[0, 0] + b1i * s[1, 0] + b2i * s[2, 0]
        map_y[yi, xi] = b0i * s[0, 1] + b1i * s[1, 1] + b2i * s[2, 1]

    gx = (map_x / (W - 1)) * 2.0 - 1.0
    gy = (map_y / (H - 1)) * 2.0 - 1.0
    return torch.from_numpy(np.stack([gx, gy], -1).astype(np.float32))


def warp_image(img: torch.Tensor, grid: torch.Tensor) -> torch.Tensor:
    B = img.shape[0]
    g = grid.unsqueeze(0).expand(B, -1, -1, -1)
    return torch.nn.functional.grid_sample(
        img, g, mode="bilinear", padding_mode="zeros", align_corners=True,
    )


# ---------------------------------------------------------------------------
# G-buffer (20 ch)
# ---------------------------------------------------------------------------

_LIP_UPPER = [61,185,40,39,37,0,267,269,270,409,291]
_LIP_LOWER = [61,146,91,181,84,17,314,405,321,375,291]
_MOUTH_IN  = [78,95,88,178,87,14,317,402,318,324,308]
_EYE_L     = [33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246,130,25,110,24,23,22,26,112]
_EYE_R     = [362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398]
_BROW_L    = [70,63,105,66,107,55,65,52,53,46]
_BROW_R    = [336,296,334,293,300,285,295,282,283,276]
_NOSE      = [1,2,3,4,5,6,45,51,48,115,131,134,102,49,220,305,275,440,281,363,360,279,294,278,344,439,122,196,3,236,198,420]


def assign_landmark_regions(N: int = 478) -> np.ndarray:
    r = np.zeros(N, dtype=np.int32)
    for i in _LIP_UPPER: r[i] = 1
    for i in _LIP_LOWER: r[i] = 2
    for i in _MOUTH_IN:  r[i] = 3
    for i in _EYE_L:     r[i] = 4
    for i in _EYE_R:     r[i] = 5
    for i in _BROW_L + _BROW_R: r[i] = 6
    for i in _NOSE:      r[i] = 7
    return r


def _vertex_normals(V: np.ndarray, simplices: np.ndarray) -> np.ndarray:
    N = np.zeros_like(V, dtype=np.float64)
    for tri in simplices:
        p = V[tri]
        N[tri] += np.cross(p[1] - p[0], p[2] - p[0])
    lens = np.linalg.norm(N, axis=1, keepdims=True)
    return (N / np.where(lens > 1e-9, lens, 1.0)).astype(np.float32)


def build_gbuffer(L_A: np.ndarray, V_out: np.ndarray, simplices: np.ndarray,
                  H: int = 128, W: int = 128, crop_size: int = 512):
    """Rasterize landmark mesh to a 20-channel G-buffer at (H, W)."""
    scale = float(H) / float(crop_size)
    src = L_A[:, :2].astype(np.float64) * scale
    dst = V_out[:, :2].astype(np.float64) * scale
    vn = _vertex_normals(V_out, simplices)
    region = assign_landmark_regions(len(L_A))
    uv = (L_A[:, :2] / float(crop_size)).astype(np.float64)

    G = np.zeros((20, H, W), dtype=np.float32)
    mask = np.zeros((H, W), dtype=np.float32)
    depth = np.zeros((H, W), dtype=np.float32)
    nrm = np.zeros((3, H, W), dtype=np.float32)
    reg = np.zeros((H, W), dtype=np.int64)
    uvm = np.zeros((2, H, W), dtype=np.float32)
    corrm = np.zeros((2, H, W), dtype=np.float32)

    for tri in simplices:
        d = dst[tri]; s = src[tri]
        z = V_out[tri, 2] * scale
        nt = vn[tri]; uvt = uv[tri]; rt = region[tri]
        rt_major = int(np.bincount(rt).argmax())

        x0 = max(0, int(np.floor(d[:, 0].min())))
        x1 = min(W, int(np.ceil(d[:, 0].max())) + 1)
        y0 = max(0, int(np.floor(d[:, 1].min())))
        y1 = min(H, int(np.ceil(d[:, 1].max())) + 1)
        if x1 <= x0 or y1 <= y0:
            continue
        xs, ys = np.meshgrid(np.arange(x0, x1, dtype=np.float32),
                             np.arange(y0, y1, dtype=np.float32))
        ax, ay = d[1, 0] - d[0, 0], d[1, 1] - d[0, 1]
        bx, by = d[2, 0] - d[0, 0], d[2, 1] - d[0, 1]
        den = ax * by - bx * ay
        if abs(den) < 1e-8:
            continue
        px, py = xs - d[0, 0], ys - d[0, 1]
        b1 = (px * by - py * bx) / den
        b2 = (ax * py - ay * px) / den
        b0 = 1.0 - b1 - b2
        inside = (b0 >= -1e-6) & (b1 >= -1e-6) & (b2 >= -1e-6)
        if not inside.any():
            continue
        xi = xs[inside].astype(np.int32); yi = ys[inside].astype(np.int32)
        b0i, b1i, b2i = b0[inside], b1[inside], b2[inside]

        depth[yi, xi] = b0i * z[0] + b1i * z[1] + b2i * z[2]
        for c in range(3):
            nrm[c, yi, xi] = b0i * nt[0, c] + b1i * nt[1, c] + b2i * nt[2, c]
        uvm[0, yi, xi] = b0i * uvt[0, 0] + b1i * uvt[1, 0] + b2i * uvt[2, 0]
        uvm[1, yi, xi] = b0i * uvt[0, 1] + b1i * uvt[1, 1] + b2i * uvt[2, 1]
        reg[yi, xi] = rt_major
        corrm[0, yi, xi] = b0i * s[0, 0] + b1i * s[1, 0] + b2i * s[2, 0]
        corrm[1, yi, xi] = b0i * s[0, 1] + b1i * s[1, 1] + b2i * s[2, 1]
        mask[yi, xi] = 1.0

    if mask.any():
        dv = depth[mask]
        dmin, dmax = float(dv.min()), float(dv.max())
        if dmax > dmin:
            depth = (depth - dmin) / (dmax - dmin) * mask

    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    flw_x = (corrm[0] - xx) / 64.0
    flw_y = (corrm[1] - yy) / 64.0

    gx = np.gradient(corrm[0], axis=1); gy = np.gradient(corrm[0], axis=0)
    hx = np.gradient(corrm[1], axis=1); hy = np.gradient(corrm[1], axis=0)
    detJ = gx * hy - gy * hx
    stretch = np.clip(np.log(np.abs(detJ) + 1e-6), -3.0, 3.0) * mask
    conf = np.exp(-(stretch ** 2) / (2 * 0.8 ** 2)) * mask

    for r in range(8):
        G[5 + r] = (reg == r).astype(np.float32) * mask

    G[0] = mask
    G[1] = depth
    G[2:5] = nrm
    G[13:15] = uvm
    G[15] = flw_x; G[16] = flw_y
    G[17] = stretch
    G[18] = conf
    return G, mask.astype(bool)


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class Stage1Dataset(Dataset):
    def __init__(self, meta_dir, crop_dir, basis_path,
                 crop_size: int = 512, gbuffer_size: int = 128,
                 realign_B: bool = True):
        self.pairs = discover_pairs(Path(meta_dir), Path(crop_dir))
        if not self.pairs:
            raise RuntimeError(f"No pairs found under {meta_dir}/{crop_dir}")
        self.crop_size = crop_size
        self.gbuffer_size = gbuffer_size
        self.realign_B = realign_B

        with np.load(basis_path, allow_pickle=True) as z:
            self.delta_B = z["delta_B"].astype(np.float32)      # (52, V, 3)

        # One-time per-subject triangulation + per-pair B→A affine
        self.simplices_per_subject: Dict[str, np.ndarray] = {}
        self.b_align: Dict[int, np.ndarray] = {}
        for i, p in enumerate(self.pairs):
            subj = p["subject"]
            with np.load(p["A_npz"]) as zA:
                A_lm = zA["landmarks"].astype(np.float32)
            if subj not in self.simplices_per_subject:
                self.simplices_per_subject[subj] = delaunay_simplices(A_lm[:, :2])
            if realign_B:
                with np.load(p["B_npz"]) as zB:
                    B_lm = zB["landmarks"].astype(np.float32)
                self.b_align[i] = _umeyama_2d(B_lm[:, :2], A_lm[:, :2])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        with np.load(p["A_npz"]) as z:
            A_lm = z["landmarks"].astype(np.float32)
            e_A = z["blendshapes"].astype(np.float32)
        with np.load(p["B_npz"]) as z:
            B_lm = z["landmarks"].astype(np.float32)
            e_B = z["blendshapes"].astype(np.float32)

        b_aff = self.b_align.get(idx)
        if b_aff is not None:
            B_lm = _apply_affine_2d(B_lm, b_aff)

        simplices = self.simplices_per_subject[p["subject"]]

        # Source image A: unmodified.
        # Target image B: warp into A's crop frame with the SAME affine as landmarks.
        A_rgb = _read_rgb(p["A_img"])
        B_rgb = _read_rgb(p["B_img"], affine_2x3=b_aff, out_size=self.crop_size)

        # V_out = A_lm + Σ_k (e_B − e_A)_k · ΔB_k
        de = (e_B - e_A).astype(np.float32)
        V_out = A_lm + np.tensordot(de, self.delta_B, axes=(0, 0)).astype(np.float32)

        # Warp A to V_out's mesh shape
        grid_512 = build_sampling_grid(
            A_lm[:, :2], V_out[:, :2], simplices,
            H=self.crop_size, W=self.crop_size,
        )

        # G-buffer at 128
        G128, mask128 = build_gbuffer(
            A_lm, V_out, simplices,
            H=self.gbuffer_size, W=self.gbuffer_size,
            crop_size=self.crop_size,
        )

        A_t = torch.from_numpy(A_rgb).permute(2, 0, 1).float() / 255.0
        B_t = torch.from_numpy(B_rgb).permute(2, 0, 1).float() / 255.0
        W512 = warp_image(A_t.unsqueeze(0), grid_512).squeeze(0)

        return {
            "A_rgb":   A_t,
            "B_rgb":   B_t,
            "W512":    W512,
            "G128":    torch.from_numpy(G128),
            "mask128": torch.from_numpy(mask128.astype(np.float32)),
            "A_lm":    torch.from_numpy(A_lm),
            "B_lm":    torch.from_numpy(B_lm),
            "e_A":     torch.from_numpy(e_A),
            "e_B":     torch.from_numpy(e_B),
            "subject": p["subject"],
            "A_stem":  p["A_stem"],
            "B_stem":  p["B_stem"],
        }

Writing stage1_data.py


In [66]:
%%writefile stage1_models.py
"""stage1_models.py — E_id (identity encoder) and P128 (128² painter)."""
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F


class FiLM(nn.Module):
    """Per-channel scale/shift conditioned on a global vector. Init to identity."""
    def __init__(self, cond_dim: int, ch: int, hidden: int = 256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(hidden, ch * 2),
        )
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        gb = self.mlp(cond)
        g, b = gb.chunk(2, dim=1)
        return x * (1.0 + g[:, :, None, None]) + b[:, :, None, None]


class ResBlock(nn.Module):
    def __init__(self, ch: int, cond_dim: int | None = None):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.n1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.n2 = nn.BatchNorm2d(ch)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.film = FiLM(cond_dim, ch) if cond_dim else None

    def forward(self, x, cond=None):
        h = self.act(self.n1(self.c1(x)))
        if self.film is not None:
            h = self.film(h, cond)
        h = self.n2(self.c2(h))
        return self.act(x + h)


class IdEncoder(nn.Module):
    """256×256 RGB → 256-d identity vector."""
    def __init__(self, out_dim: int = 256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3,   32, 3, 2, 1), nn.BatchNorm2d(32),  nn.LeakyReLU(0.1, True),
            nn.Conv2d(32,  64, 3, 2, 1), nn.BatchNorm2d(64),  nn.LeakyReLU(0.1, True),
            nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, True),
            nn.Conv2d(128,192, 3, 2, 1), nn.BatchNorm2d(192), nn.LeakyReLU(0.1, True),
            nn.Conv2d(192,256, 3, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.1, True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(256, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.stem(x)).flatten(1))


class Painter128(nn.Module):
    """
    U-Net-lite painter. Input: cat[G128 (20), W128 (3)] = 23 ch + cond (FiLM).
    Output: c0 (3, sigmoid), g0 (1, sigmoid), f128 (16, linear handoff).
    """
    def __init__(self, in_ch: int = 23, cond_dim: int = 364, f_dim: int = 16):
        super().__init__()
        self.enc0 = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1, True),
            ResBlock(64),
        )
        self.down1 = nn.Sequential(
            nn.Conv2d(64, 96, 3, stride=2, padding=1), nn.BatchNorm2d(96), nn.LeakyReLU(0.1, True),
            ResBlock(96),
        )
        self.d2_pre = nn.Sequential(
            nn.Conv2d(96, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, True),
        )
        self.d2_r1 = ResBlock(128, cond_dim)
        self.d2_r2 = ResBlock(128, cond_dim)
        self.d3_pre = nn.Sequential(
            nn.Conv2d(128, 160, 3, stride=2, padding=1), nn.BatchNorm2d(160), nn.LeakyReLU(0.1, True),
        )
        self.d3_r1 = ResBlock(160, cond_dim)
        self.d3_r2 = ResBlock(160, cond_dim)

        self.up3 = nn.Sequential(
            nn.Conv2d(160 + 128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, True),
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(128 + 96, 96, 3, padding=1), nn.BatchNorm2d(96), nn.LeakyReLU(0.1, True),
        )
        self.up1 = nn.Sequential(
            nn.Conv2d(96 + 64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1, True),
        )
        self.head = nn.Conv2d(64, 3 + 1 + f_dim, 1)

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        h0 = self.enc0(x)                       # 128², 64
        h1 = self.down1(h0)                     #  64², 96
        h2 = self.d2_pre(h1)                    #  32², 128
        h2 = self.d2_r2(self.d2_r1(h2, cond), cond)
        h3 = self.d3_pre(h2)                    #  16², 160
        h3 = self.d3_r2(self.d3_r1(h3, cond), cond)

        u3 = F.interpolate(h3, scale_factor=2, mode="bilinear", align_corners=False)
        u3 = self.up3(torch.cat([u3, h2], 1))
        u2 = F.interpolate(u3, scale_factor=2, mode="bilinear", align_corners=False)
        u2 = self.up2(torch.cat([u2, h1], 1))
        u1 = F.interpolate(u2, scale_factor=2, mode="bilinear", align_corners=False)
        u1 = self.up1(torch.cat([u1, h0], 1))

        out = self.head(u1)
        c0 = torch.sigmoid(out[:, :3])
        g0 = torch.sigmoid(out[:, 3:4])
        f128 = out[:, 4:]
        return c0, g0, f128

Writing stage1_models.py


In [67]:
%%writefile stage1_losses.py
"""stage1_losses.py — Charbonnier, LPIPS wrapper, f_id InfoNCE."""
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F


class Charbonnier(nn.Module):
    def __init__(self, eps: float = 1e-3):
        super().__init__()
        self.eps2 = eps * eps

    def forward(self, x, y):
        return torch.sqrt((x - y) ** 2 + self.eps2).mean()


def info_nce_id_loss(f_A: torch.Tensor, f_B: torch.Tensor, temperature: float = 0.1):
    """
    Within a batch, f_A[i] should be closest to f_B[i] among all f_B[j].
    Both (B, D); L2-normalized internally.
    """
    f_A = F.normalize(f_A, dim=-1)
    f_B = F.normalize(f_B, dim=-1)
    logits = f_A @ f_B.T / temperature
    labels = torch.arange(f_A.shape[0], device=f_A.device)
    return F.cross_entropy(logits, labels)

Writing stage1_losses.py


In [69]:
%%writefile train_stage1.py
"""
train_stage1.py — Stage 1: train E_id + P128 on same-identity pairs.

Pass gates (Section 17):
  * I128 beats warp-only W128 by >= 20% LPIPS@128
  * mean g0 stays in [0.3, 0.9]  (no gate collapse)

Run:
    pip install -q lpips tensorboard
    python train_stage1.py \
        --meta_dir /content/drive/MyDrive/galnp/processed/meta \
        --crop_dir /content/drive/MyDrive/galnp/processed/crops \
        --basis    /content/drive/MyDrive/expression/expression_basis.npz \
        --out_dir  /content/stage1_out \
        --epochs 200 --batch 8 --lr 2e-4
"""
from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

import lpips

from stage1_data   import Stage1Dataset
from stage1_models import IdEncoder, Painter128
from stage1_losses import Charbonnier, info_nce_id_loss


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--meta_dir", required=True)
    p.add_argument("--crop_dir", required=True)
    p.add_argument("--basis",    required=True)
    p.add_argument("--out_dir",  default="stage1_out")
    p.add_argument("--epochs",    type=int,   default=200)
    p.add_argument("--batch",     type=int,   default=8)
    p.add_argument("--lr",        type=float, default=2e-4)
    p.add_argument("--val_frac",  type=float, default=0.15)
    p.add_argument("--val_every", type=int,   default=5)
    p.add_argument("--log_every", type=int,   default=20)
    p.add_argument("--vis_every", type=int,   default=10)
    p.add_argument("--w_charb",   type=float, default=1.0)
    p.add_argument("--w_lpips",   type=float, default=0.2)
    p.add_argument("--w_id",      type=float, default=0.1)
    p.add_argument("--f_dim",     type=int,   default=256)
    p.add_argument("--f128_dim",  type=int,   default=16)
    p.add_argument("--num_workers", type=int, default=2)
    p.add_argument("--seed",      type=int,   default=0)
    return p.parse_args()


# ---------------------------------------------------------------------------

def split_pairs(n: int, val_frac: float, seed: int):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g).tolist()
    n_val = max(1, int(round(n * val_frac)))
    return perm[n_val:], perm[:n_val]


def collate(batch_list):
    out = {}
    for k in batch_list[0]:
        if isinstance(batch_list[0][k], torch.Tensor):
            out[k] = torch.stack([b[k] for b in batch_list])
        else:
            out[k] = [b[k] for b in batch_list]
    return out


def make_cond(e_B, e_A, gaze, f_id):
    return torch.cat([e_B, e_B - e_A, gaze, f_id], dim=1)


@torch.no_grad()
def evaluate(model_id, painter, loader, lpips_fn, charb, device):
    model_id.eval(); painter.eval()
    n = 0
    lp_ours = lp_warp = chr_ours = g_sum = 0.0
    for b in loader:
        A     = b["A_rgb"].to(device)
        B     = b["B_rgb"].to(device)
        W512  = b["W512"].to(device)
        G128  = b["G128"].to(device)
        e_A   = b["e_A"].to(device)
        e_B   = b["e_B"].to(device)
        Bn    = A.shape[0]

        A_256 = F.avg_pool2d(A, 2)
        B_128 = F.avg_pool2d(B, 4)
        W_128 = F.avg_pool2d(W512, 4)

        f_id = model_id(A_256)
        gaze = torch.zeros(Bn, 4, device=device)
        c0, g0, _ = painter(torch.cat([G128, W_128], 1), make_cond(e_B, e_A, gaze, f_id))
        I128 = g0 * W_128 + (1 - g0) * c0

        lp_ours  += lpips_fn(I128, B_128, normalize=True).mean().item() * Bn
        lp_warp  += lpips_fn(W_128, B_128, normalize=True).mean().item() * Bn
        chr_ours += charb(I128, B_128).item() * Bn
        g_sum    += g0.mean().item() * Bn
        n += Bn
    return {
        "lpips_ours": lp_ours / n,
        "lpips_warp": lp_warp / n,
        "charb_ours": chr_ours / n,
        "mean_g":     g_sum / max(1, n),
    }


def _tb_image_strip(A_512, B_512, W_512, I128_up, g0_up):
    """Build a single [3, H, 5*W] strip for TensorBoard logging."""
    def _to3(x):
        return x.detach().clamp(0, 1).cpu()
    A  = _to3(A_512[0])
    B  = _to3(B_512[0])
    W  = _to3(W_512[0])
    I  = _to3(I128_up[0])
    g  = _to3(g0_up[0].expand(3, -1, -1))
    return torch.cat([A, B, W, I, g], dim=2)


# ---------------------------------------------------------------------------

def main():
    args = parse_args()
    torch.manual_seed(args.seed); np.random.seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.benchmark = True

    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(args.out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(str(out_dir / "tb"))

    ds = Stage1Dataset(args.meta_dir, args.crop_dir, args.basis)
    tr_idx, va_idx = split_pairs(len(ds), args.val_frac, args.seed)
    tr = Subset(ds, tr_idx); va = Subset(ds, va_idx)

    bs = min(args.batch, len(tr))
    tr_loader = DataLoader(tr, batch_size=bs, shuffle=True,
                           num_workers=args.num_workers, collate_fn=collate, drop_last=True)
    va_loader = DataLoader(va, batch_size=min(bs, max(1, len(va))), shuffle=False,
                           num_workers=args.num_workers, collate_fn=collate)
    print(f"[stage1] {len(tr)} train pairs, {len(va)} val pairs, batch={bs}")

    model_id = IdEncoder(out_dim=args.f_dim).to(device)
    cond_dim = 52 + 52 + 4 + args.f_dim       # 364
    painter  = Painter128(in_ch=23, cond_dim=cond_dim, f_dim=args.f128_dim).to(device)
    print(f"[stage1] E_id  params = {sum(p.numel() for p in model_id.parameters())/1e6:.2f} M")
    print(f"[stage1] P128  params = {sum(p.numel() for p in painter.parameters())/1e6:.2f} M")

    lpips_fn = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
    for p in lpips_fn.parameters():
        p.requires_grad = False
    charb = Charbonnier()

    params = list(model_id.parameters()) + list(painter.parameters())
    opt = torch.optim.AdamW(params, lr=args.lr, betas=(0.9, 0.99), weight_decay=1e-4)
    total_steps = args.epochs * max(1, len(tr_loader))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=args.lr / 10)

    step = 0; t0 = time.time(); best = float("inf")

    for epoch in range(args.epochs):
        model_id.train(); painter.train()
        for b in tr_loader:
            A     = b["A_rgb"].to(device)
            B     = b["B_rgb"].to(device)
            W512  = b["W512"].to(device)
            G128  = b["G128"].to(device)
            e_A   = b["e_A"].to(device)
            e_B   = b["e_B"].to(device)
            Bn    = A.shape[0]

            A_256 = F.avg_pool2d(A, 2)
            B_128 = F.avg_pool2d(B, 4)
            W_128 = F.avg_pool2d(W512, 4)

            f_id_A = model_id(A_256)
            f_id_B = model_id(F.avg_pool2d(B, 2))
            gaze = torch.zeros(Bn, 4, device=device)

            c0, g0, _ = painter(torch.cat([G128, W_128], 1),
                                make_cond(e_B, e_A, gaze, f_id_A))
            I128 = g0 * W_128 + (1 - g0) * c0

            l_charb = charb(I128, B_128)
            l_lpips = lpips_fn(I128, B_128, normalize=True).mean()
            l_id    = info_nce_id_loss(f_id_A, f_id_B)
            loss = args.w_charb * l_charb + args.w_lpips * l_lpips + args.w_id * l_id

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); step += 1

            if step % args.log_every == 0:
                dt = time.time() - t0
                print(f"[ep {epoch:03d} st {step:06d}] "
                      f"charb={l_charb.item():.4f} lpips={l_lpips.item():.4f} "
                      f"id={l_id.item():.4f} g0={g0.mean().item():.3f} ({dt:.1f}s)")
                for k, v in [("train/charb", l_charb.item()),
                             ("train/lpips", l_lpips.item()),
                             ("train/id",    l_id.item()),
                             ("train/g0",    g0.mean().item()),
                             ("train/lr",    sched.get_last_lr()[0])]:
                    writer.add_scalar(k, v, step)
                t0 = time.time()

        if (epoch + 1) % args.val_every == 0 or epoch == args.epochs - 1:
            stats = evaluate(model_id, painter, va_loader, lpips_fn, charb, device)
            ratio = stats["lpips_ours"] / max(1e-6, stats["lpips_warp"])
            print(f"  [VAL ep {epoch:03d}] lpips_ours={stats['lpips_ours']:.4f} "
                  f"lpips_warp={stats['lpips_warp']:.4f} "
                  f"improvement={(1 - ratio) * 100:.1f}% mean_g={stats['mean_g']:.3f}")
            for k, v in [("val/lpips_ours", stats["lpips_ours"]),
                         ("val/lpips_warp", stats["lpips_warp"]),
                         ("val/charb_ours", stats["charb_ours"]),
                         ("val/mean_g",     stats["mean_g"])]:
                writer.add_scalar(k, v, step)

            if stats["lpips_ours"] < best:
                best = stats["lpips_ours"]
                torch.save({
                    "model_id": model_id.state_dict(),
                    "painter":  painter.state_dict(),
                    "opt":      opt.state_dict(),
                    "step":     step,
                    "args":     vars(args),
                    "stats":    stats,
                }, out_dir / "best.pt")
                print(f"  [VAL] new best -> {out_dir / 'best.pt'}")

        if (epoch + 1) % args.vis_every == 0:
            # Visual panel on one val batch
            model_id.eval(); painter.eval()
            with torch.no_grad():
                b = next(iter(va_loader))
                A = b["A_rgb"].to(device); B = b["B_rgb"].to(device)
                W512 = b["W512"].to(device); G128 = b["G128"].to(device)
                e_A = b["e_A"].to(device); e_B = b["e_B"].to(device)
                A_256 = F.avg_pool2d(A, 2)
                W_128 = F.avg_pool2d(W512, 4)
                f_id = model_id(A_256)
                gaze = torch.zeros(A.shape[0], 4, device=device)
                c0, g0, _ = painter(torch.cat([G128, W_128], 1),
                                    make_cond(e_B, e_A, gaze, f_id))
                I128 = g0 * W_128 + (1 - g0) * c0
                I128_up = F.interpolate(I128, scale_factor=4, mode="bilinear",
                                        align_corners=False)
                g0_up = F.interpolate(g0, scale_factor=4, mode="bilinear",
                                      align_corners=False)
                strip = _tb_image_strip(A, B, W512, I128_up, g0_up)
                writer.add_image("val/strip (A | B | W | I | g0)", strip, step)
            model_id.train(); painter.train()

        if (epoch + 1) % 20 == 0:
            torch.save({
                "model_id": model_id.state_dict(),
                "painter":  painter.state_dict(),
                "opt":      opt.state_dict(),
                "step":     step,
                "args":     vars(args),
            }, out_dir / f"ckpt_ep{epoch + 1:03d}.pt")

    writer.close()
    print("[stage1] done.")


if __name__ == "__main__":
    main()

Writing train_stage1.py


In [70]:
# 1. basis is non-trivial
import numpy as np
z = np.load('/content/drive/MyDrive/expression/expression_basis.npz')
print("delta_B", z['delta_B'].shape, "abs mean", float(np.abs(z['delta_B']).mean()))
# expect (52, 478, 3) and abs mean > 0.1

# 2. dataset returns correct shapes
import sys; sys.path.insert(0, '/content')
from stage1_data import Stage1Dataset
ds = Stage1Dataset('/content/drive/MyDrive/galnp/processed/meta',
                   '/content/drive/MyDrive/galnp/processed/crops',
                   '/content/drive/MyDrive/expression/expression_basis.npz')
s = ds[0]
for k, v in s.items():
    print(k, getattr(v, 'shape', v))

delta_B (52, 478, 3) abs mean 5.087479591369629


IndexError: arrays used as indices must be of integer (or boolean) type